### **Implementación de un DATA LAKE sobre HDFS y procesamiento de datos con HIVE**

#### **Diagrama ETL**

[![Screenshot-6.png](https://i.postimg.cc/pV05D6zc/Screenshot-6.png)](https://postimg.cc/DmXzh5C1)

#### **Introducción**

**¿Qué es un Data Lake?** Es un gran repositorio de almacenamiento de datos en bruto, sin importar su estructura, que tiene como objetivo ejecutar procesos de analítica. Nos sirve para **crear una arquitectura de gobierno de procesamiento de datos en entornos BIG DATA**.

Vamos a crear este Data Lake en el ecosistema Hadoop desplegado en el Proyecto 3 utilizando lo siguientes contenedores:

- `Hive-Server`: para hacer uso de la consola Hive
- `Namenode`: para gestionar el el sistema de archivos distribuidos HDFS

Crearemos 4 capas:

- **Paso 1.** Capturar los datos dentro de nuestro estorno de BIG DATA (`LANDING TEMP`)
- **Paso 2.** Convertirlo a un formato de rápido procesamiento (`LANDING`)    
- **Paso 3.** Transformación y almacenamiento (`UNIVERSAL`)
- **Paso 4.** Procesamiento analítico avanzado y generación de insights (`SMART`)

Recordar:
``` text
                             _________________________________________________________
                            |                                                         |      
                            | ==> Recordar que en procesos BATCH se utiliza siempre   |                                         
                            |    TODO COMPRIMIDO                                      |
                            |                                                         |                  
                            | ==> No asi en procesos REAL TIME, no debemos utilizar   |                    
                            |    compresion                                           |                                   
                            |_________________________________________________________|       
```            

#### **`Paso 1` - Creación de estructura de directorios**

In [ ]:
# Crear la estructura de directorios sobre HDFS

# Estructura de directorios LANDING_TMP
hdfs dfs -mkdir -p \
/user/proyecto/database/landing_tmp/persona \
/user/proyecto/database/landing_tmp/empresa \
/user/proyecto/database/landing_tmp/transaccion 

# Estructura de directorios LANDING
hdfs dfs -mkdir -p \
/user/proyecto/database/landing/persona \
/user/proyecto/database/landing/empresa \
/user/proyecto/database/landing/transaccion

# Estructura de directorios UNIVERSAL
hdfs dfs -mkdir -p \
/user/proyecto/database/universal/persona \
/user/proyecto/database/universal/empresa \
/user/proyecto/database/universal/transaccion \
/user/proyecto/database/universal/transaccion_enriquecida

# Estructura de directorios SMART
hdfs dfs -mkdir -p \
/user/proyecto/database/smart/transaccion_por_edad \
/user/proyecto/database/smart/transaccion_por_trabajo \
/user/proyecto/database/smart/transaccion_por_empresa 

Creamos los directorios en HDFS:

In [ ]:
# Desde el contenedor "namenode"
root@0ab30c74254e:/# hdfs dfs -mkdir -p \
> /user/proyecto/database/landing_tmp/persona \
> /user/proyecto/database/landing_tmp/empresa \
> /user/proyecto/database/landing_tmp/transaccion
root@0ab30c74254e:/#
root@0ab30c74254e:/# hdfs dfs -ls /user/proyecto/database/landing_tmp
Found 3 items
drwxr-xr-x   - root supergroup          0 2024-05-28 20:37 /user/proyecto/database/landing_tmp/empresa
drwxr-xr-x   - root supergroup          0 2024-05-28 20:37 /user/proyecto/database/landing_tmp/persona
drwxr-xr-x   - root supergroup          0 2024-05-28 20:37 /user/proyecto/database/landing_tmp/transaccion
root@0ab30c74254e:/#
root@0ab30c74254e:/# hdfs dfs -mkdir -p \
> /user/proyecto/database/landing/persona \
> /user/proyecto/database/landing/empresa \
> /user/proyecto/database/landing/transaccion
root@0ab30c74254e:/#
root@0ab30c74254e:/# hdfs dfs -ls /user/proyecto/database/landing
Found 3 items
drwxr-xr-x   - root supergroup          0 2024-05-28 20:41 /user/proyecto/database/landing/empresa
drwxr-xr-x   - root supergroup          0 2024-05-28 20:41 /user/proyecto/database/landing/persona
drwxr-xr-x   - root supergroup          0 2024-05-28 20:41 /user/proyecto/database/landing/transaccion
root@0ab30c74254e:/#
root@0ab30c74254e:/# hdfs dfs -mkdir -p \
> /user/proyecto/database/universal/persona \
> /user/proyecto/database/universal/empresa \
> /user/proyecto/database/universal/transaccion \
> /user/proyecto/database/universal/transaccion_enriquecida
root@0ab30c74254e:/#
root@0ab30c74254e:/# hdfs dfs -ls /user/proyecto/database/universal
Found 4 items
drwxr-xr-x   - root supergroup          0 2024-05-28 20:46 /user/proyecto/database/universal/empresa
drwxr-xr-x   - root supergroup          0 2024-05-28 20:46 /user/proyecto/database/universal/persona
drwxr-xr-x   - root supergroup          0 2024-05-28 20:46 /user/proyecto/database/universal/transaccion
drwxr-xr-x   - root supergroup          0 2024-05-28 20:46 /user/proyecto/database/universal/transaccion_enriquecida
root@0ab30c74254e:/#
root@0ab30c74254e:/# hdfs dfs -mkdir -p \
> /user/proyecto/database/smart/transaccion_por_edad \
> /user/proyecto/database/smart/transaccion_por_trabajo \
> /user/proyecto/database/smart/transaccion_por_empresa
root@0ab30c74254e:/#
root@0ab30c74254e:/# hdfs dfs -ls /user/proyecto/database/smart
Found 3 items
drwxr-xr-x   - root supergroup          0 2024-05-28 20:47 /user/proyecto/database/smart/transaccion_por_edad
drwxr-xr-x   - root supergroup          0 2024-05-28 20:47 /user/proyecto/database/smart/transaccion_por_empresa
drwxr-xr-x   - root supergroup          0 2024-05-28 20:47 /user/proyecto/database/smart/transaccion_por_trabajo
root@0ab30c74254e:/#

Subimos los archivos de esquemas que se utilizaran en la creacion de la capa LANDING

In [ ]:
hdfs dfs -mkdir -p /user/proyecto/schema/landing

hdfs dfs -put \
/home/hadoop/archivos/persona.avsc \
/home/hadoop/archivos/empresa.avsc \
/home/hadoop/archivos/transaccion.avsc \
/user/proyecto/schema/landing

In [ ]:
# Creamos la ruta desde el contenedor "namenode"
root@0ab30c74254e:/# hdfs dfs -ls /user/proyecto/schema
Found 1 items
drwxr-xr-x   - root supergroup          0 2024-05-28 21:14 /user/proyecto/schema/landing
root@0ab30c74254e:/#

In [ ]:
# Copio los archivos de esquema desde mi sistema windows hacia el contenedor "namenode"
C:\Users\Alfonso>docker cp empresa.avsc namenode:/home/hadoop/archivos
Successfully copied 2.05kB to namenode:/home/hadoop/archivos

C:\Users\Alfonso>docker cp persona.avsc namenode:/home/hadoop/archivos
Successfully copied 2.05kB to namenode:/home/hadoop/archivos

C:\Users\Alfonso>docker cp transaccion.avsc namenode:/home/hadoop/archivos
Successfully copied 2.05kB to namenode:/home/hadoop/archivos

C:\Users\Alfonso>

In [ ]:
# Desde el contenedor "namenode" verifico que los archivos hayan sido recibidos
root@0ab30c74254e:/home/hadoop/archivos# ls -l
total 6660
-rw-r--r-- 1 root root 6799918 May 28 15:45 athlete_events.csv
-rwxr-xr-x 1 root root     160 Feb 24  2020 empresa.avsc        # <---
-rw-r--r-- 1 root root    3595 May 28 16:20 noc_regions.csv     # <---
-rwxr-xr-x 1 root root     498 Feb 24  2020 persona.avsc        # <---
-rwxr-xr-x 1 root root     226 Feb 24  2020 transaccion.avsc
root@0ab30c74254e:/home/hadoop/archivos#

In [ ]:
# Copio los archivos hacia el directorio "/user/proyecto/schema/landing" en HDFS
hdfs dfs -put \
/home/hadoop/archivos/persona.avsc \
/home/hadoop/archivos/empresa.avsc \
/home/hadoop/archivos/transaccion.avsc \
/user/proyecto/schema/landing

In [ ]:
# Copio los archivos hacia el directorio "/user/proyecto/schema/landing" en HDFS
root@0ab30c74254e:/home/hadoop/archivos# hdfs dfs -put \
> /home/hadoop/archivos/persona.avsc \
> /home/hadoop/archivos/empresa.avsc \
> /home/hadoop/archivos/transaccion.avsc \
> /user/proyecto/schema/landing
root@0ab30c74254e:/home/hadoop/archivos#
root@0ab30c74254e:/home/hadoop/archivos# hdfs dfs -ls /user/proyecto/schema/landing
Found 3 items
-rw-r--r--   3 root supergroup        160 2024-05-28 21:28 /user/proyecto/schema/landing/empresa.avsc
-rw-r--r--   3 root supergroup        498 2024-05-28 21:28 /user/proyecto/schema/landing/persona.avsc
-rw-r--r--   3 root supergroup        226 2024-05-28 21:28 /user/proyecto/schema/landing/transaccion.avsc
root@0ab30c74254e:/home/hadoop/archivos#

#### **`Paso 2` - Crear bases de datos**

In [ ]:
CREATE DATABASE IF NOT EXISTS LANDING_TMP LOCATION '/user/proyecto/database/landing_tmp';
CREATE DATABASE IF NOT EXISTS LANDING LOCATION '/user/proyecto/database/landing';
CREATE DATABASE IF NOT EXISTS UNIVERSAL LOCATION '/user/proyecto/database/universal';
CREATE DATABASE IF NOT EXISTS SMART LOCATION '/user/proyecto/database/smart';

In [ ]:
# Desde el contenedor "hive-server"
# En la consola de HIVE, adaptar y ejecutar:
hive> CREATE DATABASE IF NOT EXISTS LANDING_TMP LOCATION '/user/proyecto/database/landing_tmp';
OK
Time taken: 0.709 seconds
hive> CREATE DATABASE IF NOT EXISTS LANDING LOCATION '/user/proyecto/database/landing';
OK
Time taken: 0.102 seconds
hive> CREATE DATABASE IF NOT EXISTS UNIVERSAL LOCATION '/user/proyecto/database/universal';
OK
Time taken: 0.079 seconds
hive> CREATE DATABASE IF NOT EXISTS SMART LOCATION '/user/proyecto/database/smart';
OK
Time taken: 0.049 seconds
hive>

In [ ]:
# Desde el contenedor "namenode"
# Los directorios corresponden a las bases de datos
# Estas rutas de directorios la habiamos creado en el paso 1
# Sin embargo, es ahora cuando pasan a ser una base de datos
# No llevan la extensión ".db"
root@0ab30c74254e:/# hdfs dfs -ls /user/proyecto/database
Found 4 items
drwxr-xr-x   - root supergroup          0 2024-05-28 20:41 /user/proyecto/database/landing
drwxr-xr-x   - root supergroup          0 2024-05-28 20:37 /user/proyecto/database/landing_tmp
drwxr-xr-x   - root supergroup          0 2024-05-28 20:47 /user/proyecto/database/smart
drwxr-xr-x   - root supergroup          0 2024-05-28 20:46 /user/proyecto/database/universal
root@0ab30c74254e:/#

#### **`Paso 3` - Crear la capa LANDING_TMP**

En este paso inicial, los datos son adquiridos desde diversas fuentes y capturados en la capa LANDING_TMP del Data Lake. Estos datos pueden provenir de sistemas transaccionales, dispositivos IoT, archivos de registro, redes sociales, fuentes externas, entre otros. La capa LANDING_TMP actúa como un área temporal donde se almacenan los datos en su forma bruta o sin procesar.

Es importante aplicar mecanismos de captura de datos que garanticen la integridad y la disponibilidad de los datos durante el proceso de ingestión. Esto puede incluir técnicas como la replicación de datos, el cambio de datos por lotes o en tiempo real, y la validación de datos para garantizar su calidad y precisión.

- Para no complicar la explicacion le vamos a crear una tabla de **TEXTO PLANO**
- Y como es **data estructurada** le creamos una tabla **HIVE** por encima
- Dentro de la capa `LANDING TMP`, el unico tipo de dato que se utiliza es el **STRING** al igual que en la capa `LADING`. El objetivo de esto es capturar la data aunque tenga tenga errores.

##### Carga de archivos de sistema windows a sistema local contenedor

In [ ]:
# Copio los archivos que utilizaremos para la carga de las tablas desde mi sistema windows hacia el contenedor "namenode"
C:\Users\Alfonso>docker cp transacciones.data namenode:/home/hadoop/archivos
Successfully copied 5.13MB to namenode:/home/hadoop/archivos

C:\Users\Alfonso>docker cp empresa.data namenode:/home/hadoop/archivos
Successfully copied 2.05kB to namenode:/home/hadoop/archivos

C:\Users\Alfonso>docker cp persona.data namenode:/home/hadoop/archivos
Successfully copied 9.22kB to namenode:/home/hadoop/archivos

C:\Users\Alfonso>

In [ ]:
# Desde el contenedor "namenode" verifico que los archivos hayan sido recibidos
root@0ab30c74254e:/home/hadoop/archivos# ls -l
total 11684
-rw-r--r-- 1 root root 6799918 May 28 15:45 athlete_events.csv
-rwxr-xr-x 1 root root     160 Feb 24  2020 empresa.avsc
-rwxr-xr-x 1 root root     105 Jun 23  2018 empresa.data        # <---
-rw-r--r-- 1 root root    3595 May 28 16:20 noc_regions.csv
-rwxr-xr-x 1 root root     498 Feb 24  2020 persona.avsc
-rwxr-xr-x 1 root root    7282 Jun 23  2018 persona.data        # <---
-rwxr-xr-x 1 root root     226 Feb 24  2020 transaccion.avsc
-rwxr-xr-x 1 root root 5129134 Dec 15  2018 transacciones.data  # <---
root@0ab30c74254e:/home/hadoop/archivos#

##### `PERSONA`

In [ ]:
                                              ------------
                                                 PERSONA    
                                              ------------
-- En la consola de HIVE, adaptar y ejecutar
CREATE TABLE LANDING_TMP.PERSONA(
ID STRING,
NOMBRE STRING,
TELEFONO STRING,
CORREO STRING,
FECHA_INGRESO STRING,
EDAD STRING,
SALARIO STRING,
ID_EMPRESA STRING
)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY '|'
LINES TERMINATED BY '\n'
STORED AS TEXTFILE
LOCATION '/user/proyecto/database/landing_tmp/persona';

-- Subida de datos
-- En la consola HADOOP, adaptar y ejecutar
hdfs dfs -put /home/hadoop/archivos/persona.data /user/proyecto/database/landing_tmp/persona

-- O desde HIVE utilizando (es lo mismo)
-- En nuestro caso, yo estoy usando dos contenedores para trabajar: "namenode" y "hive-server"
-- Los archivos locales los estoy manejando en el contenedor "namenode" y la consola hive en "hive-server"
-- Por tanto, no podria cargar un archivo local que se encuentra en el contenedor "namenode" desde el contenedor "hive-server"
-- ¿Que opciones tengo? Subir todos estos archivos al contenedor "hive-server" y desde ahi utilizarlos para cargar las tablas de manera local: LOAD DATA LOCAL INPATH
-- O desde el contenedor "namenode" copiarlos al HDFS y desde ahi cargar las tablas con LOAD DATA INPATH 
LOAD DATA LOCAL INPATH '/home/hadoop/archivos/persona.data' INTO TABLE LANDING_TMP.PERSONA;

In [ ]:
# Desde el contenedor "hive-server"
hive> CREATE TABLE LANDING_TMP.PERSONA(
    > ID STRING,
    > NOMBRE STRING,
    > TELEFONO STRING,
    > CORREO STRING,
    > FECHA_INGRESO STRING,
    > EDAD STRING,
    > SALARIO STRING,
    > ID_EMPRESA STRING
    > )
    > ROW FORMAT DELIMITED
    > FIELDS TERMINATED BY '|'
    > LINES TERMINATED BY '\n'
    > STORED AS TEXTFILE
    > LOCATION '/user/proyecto/database/landing_tmp/persona'; # Esta ruta la habiamos creado solo como directorio
OK                                                                       # Sin embargo, es ahora al indicarla al momento de 
Time taken: 0.372 seconds                                                # crear la tabla que pasa a ser la tabla "persona"           
hive>

In [ ]:
# Desde el contenedor "namenode"
# De esta forma estamos cargando la tabla "LANDING_TMP.PERSONA" con datos del archivo "persona.data"
root@0ab30c74254e:/# hdfs dfs -put /home/hadoop/archivos/persona.data /user/proyecto/database/landing_tmp/persona
root@0ab30c74254e:/#
root@0ab30c74254e:/# hdfs dfs -ls /user/proyecto/database/landing_tmp/persona
Found 1 items
-rw-r--r--   3 root supergroup       7282 2024-05-28 21:53 /user/proyecto/database/landing_tmp/persona/persona.data
root@0ab30c74254e:/#

In [ ]:
# En la consola HIVE o en HUE, verificar que existan los datos en la tabla
# Se cargara el "ENCABEZADO" como un registro más, eso no es correcto, sin embargo, este tipo de  errores se CORRIGEN EN LA ETAPA "UNIVERSAL"
# Repetir los pasos anteriores para las tablas "EMPRESA" y "TRANSACCION" 
hive> SELECT * FROM LANDING_TMP.PERSONA LIMIT 10;
OK
+-------------+-----------------+-------------------+-----------------------------------------+------------------------+---------------+------------------+---------------------+
| persona.id  | persona.nombre  | persona.telefono  |             persona.correo              | persona.fecha_ingreso  | persona.edad  | persona.salario  | persona.id_empresa  |
+-------------+-----------------+-------------------+-----------------------------------------+------------------------+---------------+------------------+---------------------+
| ID          | NOMBRE          | TELEFONO          | CORREO                                  | FECHA_INGRESO          | EDAD          | SALARIO          | ID_EMPRESA          |
| 1           | Carl            | 1-745-633-9145    | arcu.Sed.et@ante.co.uk                  | 2004-04-23             | 32            | 20095            | 5                   |
| 2           | Priscilla       | 155-2498          | Donec.egestas.Aliquam@volutpatnunc.edu  | 2019-02-17             | 34            | 9298             | 2                   |
| 3           | Jocelyn         | 1-204-956-8594    | amet.diam@lobortis.co.uk                | 2002-08-01             | 27            | 10853            | 3                   |
| 4           | Aidan           | 1-719-862-9385    | euismod.et.commodo@nibhlaciniaorci.edu  | 2018-11-06             | 29            | 3387             | 10                  |
| 5           | Leandra         | 839-8044          | at@pretiumetrutrum.com                  | 2002-10-10             | 41            | 22102            | 1                   |
| 6           | Bert            | 797-4453          | a.felis.ullamcorper@arcu.org            | 2017-04-25             | 70            | 7800             | 7                   |
| 7           | Mark            | 1-680-102-6792    | Quisque.ac@placerat.ca                  | 2006-04-21             | 52            | 8112             | 5                   |
| 8           | Jonah           | 214-2975          | eu.ultrices.sit@vitae.ca                | 2017-10-07             | 23            | 17040            | 5                   |
| 9           | Hanae           | 935-2277          | eu@Nunc.ca                              | 2003-05-25             | 69            | 6834             | 3                   |
+-------------+-----------------+-------------------+-----------------------------------------+------------------------+---------------+------------------+---------------------+

##### `EMPRESA`

In [ ]:
                                              ------------
                                                 EMPRESA    
                                              ------------
-- En la consola de HIVE, adaptar y ejecutar
CREATE TABLE LANDING_TMP.EMPRESA(
ID STRING,
NOMBRE STRING
) 
ROW FORMAT DELIMITED
FIELDS TERMINATED BY '|'
LINES TERMINATED BY '\n'
STORED AS TEXTFILE
LOCATION '/user/proyecto/database/landing_tmp/empresa';

-- Subida de datos
-- En la consola HADOOP, adaptar y ejecutar
hdfs dfs -put /home/hadoop/archivos/empresa.data /user/proyecto/database/landing_tmp/empresa

-- O desde HIVE utilizando (es lo mismo)
-- En nuestro caso, yo estoy usando dos contenedores para trabajar: "namenode" y "hive-server"
-- Los archivos locales los estoy manejando en el contenedor "namenode" y la consola hive en "hive-server"
-- Por tanto, no podria cargar un archivo local que se encuentra en el contenedor "namenode" desde el contenedor "hive-server"
-- ¿Que opciones tengo? Subir todos estos archivos al contenedor "hive-server" y desde ahi utilizarlos para cargar las tablas de manera local: LOAD DATA LOCAL INPATH
-- O desde el contenedor "namenode" copiarlos al HDFS y desde ahi cargar las tablas con LOAD DATA INPATH 
LOAD DATA LOCAL INPATH '/home/hadoop/data/empresa.data' INTO TABLE LANDING_TMP.EMPRESA;  

In [ ]:
# Desde el contenedor "hive-server"
hive> CREATE TABLE LANDING_TMP.EMPRESA(
    > ID STRING,
    > NOMBRE STRING
    > )
    > ROW FORMAT DELIMITED
    > FIELDS TERMINATED BY '|'
    > LINES TERMINATED BY '\n'
    > STORED AS TEXTFILE
    > LOCATION '/user/proyecto/database/landing_tmp/empresa'; # Esta ruta la habiamos creado solo como directorio
OK                                                                       # Sin embargo, es ahora al indicarla al momento de 
Time taken: 0.129 seconds                                                # crear la tabla que pasa a ser la tabla "empresa"           
hive>

In [ ]:
# Desde el contenedor "hi-server"
# De esta forma podemos cargar la tabla "LANDING_TMP.EMPRESA" con datos del archivo "empresa.data"
root@0ab30c74254e:/# hdfs dfs -put /home/hadoop/archivos/empresa.data /user/proyecto/database/landing_tmp/empresa
root@0ab30c74254e:/#
root@0ab30c74254e:/# hdfs dfs -ls /user/proyecto/database/landing_tmp/empresa
Found 1 items
-rw-r--r--   3 root supergroup        105 2024-05-28 22:50 /user/proyecto/database/landing_tmp/empresa/empresa.data
root@0ab30c74254e:/#

In [ ]:
# En la consola HIVE o en HUE, verificar que existan los datos en la tabla
# Se cargara el "ENCABEZADO" como un registro más, eso no es correcto, sin embargo, este tipo de  errores se CORRIGEN EN LA ETAPA "UNIVERSAL"
# Repetir los pasos anteriores para las tablas "TRANSACCION" 
hive> SELECT * FROM LANDING_TMP.EMPRESA LIMIT 10;
OK
+-------------+-----------------+
| empresa.id  | empresa.nombre  |
+-------------+-----------------+
| ID          | NOMBRE          |
| 1           | Walmart         |
| 2           | Microsoft       |
| 3           | Apple           |
| 4           | Toyota          |
| 5           | Amazon          |
| 6           | Google          |
| 7           | Samsung         |
| 8           | HP              |
| 9           | IBM             |
+-------------+-----------------+

##### `TRANSACCION`

In [ ]:
                                              ------------
                                                 EMPRESA    
                                              ------------
-- En la consola de HIVE, adaptar y ejecutar
CREATE TABLE LANDING_TMP.TRANSACCION(
ID_PERSONA STRING,
ID_EMPRESA STRING,
MONTO STRING,
FECHA STRING  <------------------------------------- Agrego este campo, siendo que no existe en el archivo de definicion de esquema ¿Por que?
)                                                    Porque al crear una tabla particionada, no se incluye el campo particionado, en este caso,
ROW FORMAT DELIMITED                                 el campo FECHA (En la capa LANDING)
FIELDS TERMINATED BY '|'
LINES TERMINATED BY '\n'
STORED AS TEXTFILE
LOCATION '/user/proyecto/database/landing_tmp/transaccion';

-- Subida de datos
-- En la consola HADOOP, adaptar y ejecutar
hdfs dfs -put /home/hadoop/archivos/transacciones.data /user/proyecto/database/landing_tmp/transaccion 

-- O desde HIVE utilizando (es lo mismo)
-- En nuestro caso, yo estoy usando dos contenedores para trabajar: "namenode" y "hive-server"
-- Los archivos locales los estoy manejando en el contenedor "namenode" y la consola hive en "hive-server"
-- Por tanto, no podria cargar un archivo local que se encuentra en el contenedor "namenode" desde el contenedor "hive-server"
-- ¿Que opciones tengo? Subir todos estos archivos al contenedor "hive-server" y desde ahi utilizarlos para cargar las tablas de manera local: LOAD DATA LOCAL INPATH
-- O desde el contenedor "namenode" copiarlos al HDFS y desde ahi cargar las tablas con LOAD DATA INPATH 
LOAD DATA LOCAL INPATH '/home/hadoop/data/transacciones.data' INTO TABLE LANDING_TMP.TRANSACCION;

In [ ]:
# Desde el contenedor "hive-server"
hive> CREATE TABLE LANDING_TMP.TRANSACCION(
    > ID_PERSONA STRING,
    > ID_EMPRESA STRING,
    > MONTO STRING,
    > FECHA STRING
    > )
    > ROW FORMAT DELIMITED
    > FIELDS TERMINATED BY '|'
    > LINES TERMINATED BY '\n'
    > STORED AS TEXTFILE
    > LOCATION '/user/proyecto/database/landing_tmp/transaccion'; # Esta ruta la habiamos creado solo como directorio
OK                                                                           # Sin embargo, es ahora al indicarla al momento de 
Time taken: 0.098 seconds                                                    # crear la tabla que pasa a ser la tabla "transaccion"           
hive>

In [ ]:
# Desde el contenedor "hi-server"
# De esta forma podemos cargar la tabla "LANDING_TMP.TRANSACCION" con datos del archivo "transaccion.data"
root@0ab30c74254e:/# hdfs dfs -put /home/hadoop/archivos/transacciones.data /user/proyecto/database/landing_tmp/transaccion
root@0ab30c74254e:/#
root@0ab30c74254e:/# hdfs dfs -ls /user/proyecto/database/landing_tmp/transaccion
Found 1 items
-rw-r--r--   3 root supergroup    5129134 2024-05-28 23:05 /user/proyecto/database/landing_tmp/transaccion/transacciones.data
root@0ab30c74254e:/#

In [ ]:
# En la consola HIVE o en HUE, verificar que existan los datos en la tabla
# Se cargara el "ENCABEZADO" como un registro más, eso no es correcto, sin embargo, este tipo de  errores se CORRIGEN EN LA ETAPA "UNIVERSAL"
hive> SELECT * FROM LANDING_TMP.TRANSACCION LIMIT 10;
OK
+-------------------------+-------------------------+--------------------+--------------------+
| transaccion.id_persona  | transaccion.id_empresa  | transaccion.monto  | transaccion.fecha  |
+-------------------------+-------------------------+--------------------+--------------------+
| ID_PERSONA              | ID_EMPRESA              | MONTO              | FECHA              |
| 18                      | 3                       | 1383               | 2018-01-21         |
| 30                      | 6                       | 2331               | 2018-01-21         |
| 47                      | 2                       | 2280               | 2018-01-21         |
| 28                      | 1                       | 730                | 2018-01-21         |
| 91                      | 4                       | 3081               | 2018-01-21         |
| 74                      | 8                       | 2409               | 2018-01-21         |
| 41                      | 2                       | 3754               | 2018-01-22         |
| 42                      | 9                       | 4079               | 2018-01-22         |
| 24                      | 6                       | 4475               | 2018-01-22         |
+-------------------------+-------------------------+--------------------+--------------------+

#### **`Paso 4` - Crear la capa LANDING**

Una vez que los datos han sido capturados en la capa LANDING_TMP, pasan por un proceso de transformación y limpieza para prepararlos para su almacenamiento en la capa LANDING del Data Lake. Durante esta etapa, los datos pueden ser normalizados, estructurados, enriquecidos y formateados según sea necesario para su posterior procesamiento y análisis.

La capa LANDING actúa como un repositorio centralizado de datos que han sido transformados y están listos para su uso en el Data Lake. Estos datos suelen estar organizados en un formato optimizado para el procesamiento posterior y pueden ser accesibles para diferentes equipos y aplicaciones en la organización.

Es importante aplicar prácticas de gobernanza de datos en esta etapa para garantizar la calidad, la integridad y la seguridad de los datos en la capa LANDING. Esto puede incluir la implementación de políticas de control de acceso, la encriptación de datos, la monitorización de la calidad de los datos y la gestión de metadatos para asegurar la trazabilidad y la transparencia en el uso de los datos.

En resumen, los pasos 1 y 2 en la construcción de un Data Lake involucran la captura de datos en su forma bruta en la capa LANDING_TMP, seguida de su transformación y almacenamiento en la capa LANDING en un formato preparado para su procesamiento y análisis posterior. Estos pasos son fundamentales para garantizar la disponibilidad y la calidad de los datos en el Data Lake.

- En la capa `LANDING` vamos a convertir todo, con errores incluidos. No se excluye nada. Para efectos del ejemplo, convertiremos a formato **AVRO**.
- Dentro de la capa `LANDING` el unico tipo de dato que se utiliza es el **STRING**. No se debe utilizar otro. Esto con el fin de aceptar cualquier tipo de dato.
- Gracias a la capa `LANDING` se tiene una flexibilidad para responder rapidamente a los cambios en las fuentes. 

##### `PERSONA`

In [ ]:
                                              ------------
                                                 PERSONA    
                                              ------------
------------------
ACTIVAR COMPRESION
------------------

-- Desde HIVE, activamos la compresión y el formato SNAPPY
SET hive.exec.compress.output=true;
SET avro.output.codec=snappy;

In [ ]:
# Activamos la compresión SNAPPY desde consola Hive
hive> SET hive.exec.compress.output=true;
hive> SET avro.output.codec=snappy;
hive>

In [ ]:
-----------
CREAR TABLA
-----------

-- Desde HIVE, creamos la tabla PERSONA 
CREATE TABLE LANDING.PERSONA
STORED AS AVRO
LOCATION '/user/proyecto/database/landing/persona'
TBLPROPERTIES (
'avro.schema.url'='hdfs:///user/proyecto/schema/landing/persona.avsc',
'avro.output.codec'='snappy'
);

In [ ]:
# Desde la consola Hive
hive> CREATE TABLE LANDING.PERSONA
    > STORED AS AVRO
    > LOCATION '/user/proyecto/database/landing/persona'
    > TBLPROPERTIES (
    > 'avro.schema.url'='hdfs:///user/proyecto/schema/landing/persona.avsc',
    > 'avro.output.codec'='snappy'
    > );
OK
Time taken: 0.195 seconds
hive>

In [ ]:
------------------------------
INSERCION DE DATOS EN LA TABLA
------------------------------

-- Desde HIVE, ejecutamos la inserción de datos desde la tabla "LANDING_TMP" a la tabla "LANDING"
INSERT OVERWRITE TABLE LANDING.PERSONA SELECT * FROM LANDING_TMP.PERSONA;

In [ ]:
# Realizamos la inserción de datos hacia la tabla "landing.persona" desde la tabla "landing_tmp.persona"
hive> INSERT OVERWRITE TABLE LANDING.PERSONA SELECT * FROM LANDING_TMP.PERSONA;
WARNING: Hive-on-MR is deprecated in Hive 2 and may not be available in the future versions. Consider using a different execution engine (i.e. spark, tez) or using Hive 1.X releases.
Query ID = root_20240529180131_57bc5a03-c3b0-429a-9300-ff58b1959dcf
Total jobs = 3
Launching Job 1 out of 3
Number of reduce tasks is set to 0 since there's no reduce operator
Job running in-process (local Hadoop)
2024-05-29 18:01:33,994 Stage-1 map = 100%,  reduce = 0%
Ended Job = job_local1476198451_0001
Stage-4 is selected by condition resolver.
Stage-3 is filtered out by condition resolver.
Stage-5 is filtered out by condition resolver.
Moving data to directory hdfs://namenode:8020/user/proyecto/database/landing/persona/.hive-staging_hive_2024-05-29_18-01-31_524_5920763197649682191-1/-ext-10000
Loading data to table landing.persona
MapReduce Jobs Launched:
Stage-Stage-1:  HDFS Read: 19065 HDFS Write: 6626 SUCCESS
Total MapReduce CPU Time Spent: 0 msec
OK
Time taken: 4.1 seconds
hive>

In [ ]:
-- Desde HIVE, verificamos que se hayan insertado los datos en la tabla "LANDING"
SELECT * FROM LANDING.PERSONA LIMIT 10;  

In [ ]:
hive> SELECT * FROM LANDING.PERSONA LIMIT 10;
OK
+-------------+-----------------+-------------------+-----------------------------------------+------------------------+---------------+------------------+---------------------+
| persona.id  | persona.nombre  | persona.telefono  |             persona.correo              | persona.fecha_ingreso  | persona.edad  | persona.salario  | persona.id_empresa  |
+-------------+-----------------+-------------------+-----------------------------------------+------------------------+---------------+------------------+---------------------+
| ID          | NOMBRE          | TELEFONO          | CORREO                                  | FECHA_INGRESO          | EDAD          | SALARIO          | ID_EMPRESA          |
| 1           | Carl            | 1-745-633-9145    | arcu.Sed.et@ante.co.uk                  | 2004-04-23             | 32            | 20095            | 5                   |
| 2           | Priscilla       | 155-2498          | Donec.egestas.Aliquam@volutpatnunc.edu  | 2019-02-17             | 34            | 9298             | 2                   |
| 3           | Jocelyn         | 1-204-956-8594    | amet.diam@lobortis.co.uk                | 2002-08-01             | 27            | 10853            | 3                   |
| 4           | Aidan           | 1-719-862-9385    | euismod.et.commodo@nibhlaciniaorci.edu  | 2018-11-06             | 29            | 3387             | 10                  |
| 5           | Leandra         | 839-8044          | at@pretiumetrutrum.com                  | 2002-10-10             | 41            | 22102            | 1                   |
| 6           | Bert            | 797-4453          | a.felis.ullamcorper@arcu.org            | 2017-04-25             | 70            | 7800             | 7                   |
| 7           | Mark            | 1-680-102-6792    | Quisque.ac@placerat.ca                  | 2006-04-21             | 52            | 8112             | 5                   |
| 8           | Jonah           | 214-2975          | eu.ultrices.sit@vitae.ca                | 2017-10-07             | 23            | 17040            | 5                   |
| 9           | Hanae           | 935-2277          | eu@Nunc.ca                              | 2003-05-25             | 69            | 6834             | 3                   |
+-------------+-----------------+-------------------+-----------------------------------------+------------------------+---------------+------------------+---------------------+

##### `EMPRESA`

In [ ]:
                                              ------------
                                                 EMPRESA    
                                              ------------
------------------
ACTIVAR COMPRESION
------------------

-- Desde HIVE, activamos la compresión y el formato SNAPPY (Ya lo hicimos, pero para recordar)
SET hive.exec.compress.output=true;
SET avro.output.codec=snappy;

In [ ]:
-----------
CREAR TABLA
-----------

-- Desde HIVE, creamos la tabla EMPRESA 
CREATE TABLE LANDING.EMPRESA
STORED AS AVRO
LOCATION '/user/proyecto/database/landing/empresa'
TBLPROPERTIES (
'avro.schema.url'='hdfs:///user/proyecto/schema/landing/empresa.avsc',
'avro.output.codec'='snappy'
);

In [ ]:
# Desde la consola Hive
hive> CREATE TABLE LANDING.EMPRESA
    > STORED AS AVRO
    > LOCATION '/user/proyecto/database/landing/empresa'
    > TBLPROPERTIES (
    > 'avro.schema.url'='hdfs:///user/proyecto/schema/landing/empresa.avsc',
    > 'avro.output.codec'='snappy'
    > );
OK
Time taken: 0.098 seconds
hive>

In [ ]:
------------------------------
INSERCION DE DATOS EN LA TABLA
------------------------------

-- Desde HIVE, ejecutamos la inserción de datos desde la tabla "LANDING_TMP" a la tabla "LANDING"
INSERT OVERWRITE TABLE LANDING.EMPRESA SELECT * FROM LANDING_TMP.EMPRESA;

In [ ]:
# Realizamos la inserción de datos hacia la tabla "landing.empresa" desde la tabla "landing_tmp.empresa"
hive> INSERT OVERWRITE TABLE LANDING.EMPRESA SELECT * FROM LANDING_TMP.EMPRESA;
WARNING: Hive-on-MR is deprecated in Hive 2 and may not be available in the future versions. Consider using a different execution engine (i.e. spark, tez) or using Hive 1.X releases.
Query ID = root_20240529180518_13c26703-d6e2-47be-b347-b36becd3ec6e
Total jobs = 3
Launching Job 1 out of 3
Number of reduce tasks is set to 0 since there's no reduce operator
Job running in-process (local Hadoop)
2024-05-29 18:05:19,992 Stage-1 map = 100%,  reduce = 0%
Ended Job = job_local1892201689_0002
Stage-4 is selected by condition resolver.
Stage-3 is filtered out by condition resolver.
Stage-5 is filtered out by condition resolver.
Moving data to directory hdfs://namenode:8020/user/proyecto/database/landing/empresa/.hive-staging_hive_2024-05-29_18-05-18_286_6185861273236917558-1/-ext-10000
Loading data to table landing.empresa
MapReduce Jobs Launched:
Stage-Stage-1:  HDFS Read: 37471 HDFS Write: 7005 SUCCESS
Total MapReduce CPU Time Spent: 0 msec
OK
Time taken: 2.1 seconds
hive>

In [ ]:
-- Desde HIVE, verificamos que se hayan insertado los datos en la tabla "LANDING"
SELECT * FROM LANDING.EMPRESA LIMIT 10; 

In [ ]:
hive> SELECT * FROM LANDING.EMPRESA LIMIT 10;
OK
+-------------+-----------------+
| empresa.id  | empresa.nombre  |
+-------------+-----------------+
| ID          | NOMBRE          |
| 1           | Walmart         |
| 2           | Microsoft       |
| 3           | Apple           |
| 4           | Toyota          |
| 5           | Amazon          |
| 6           | Google          |
| 7           | Samsung         |
| 8           | HP              |
| 9           | IBM             |
+-------------+-----------------+

##### `TRANSACCION`

In [ ]:
                                            ----------------
                                               TRANSACCION    
                                            ----------------
------------------
ACTIVAR COMPRESION
------------------

-- Desde HIVE, activamos la compresión y el formato SNAPPY (Ya lo hicimos, pero para recordar)
SET hive.exec.compress.output=true;
SET avro.output.codec=snappy;

In [ ]:
---------------------------------
ACTIVAR PARTICIONAMIENTO DINAMICO
---------------------------------

SET hive.exec.dynamic.partition=true;
SET hive.exec.dynamic.partition.mode=nonstrict;

In [ ]:
hive> SET hive.exec.dynamic.partition=true;
hive> SET hive.exec.dynamic.partition.mode=nonstrict;
hive>

In [ ]:
-----------
CREAR TABLA
-----------

Queremos crear una tabla particionada. Las particiones NO SON FLEXIBLES aunque esten en un AVRO, son
ESTATICAS. Porque a nivel de HDFS, una particion es un Subdirectorio de HDFS, asi que es algo estatico.

-- Desde HIVE, creamos la tabla TRANSACCION 
CREATE TABLE LANDING.TRANSACCION
PARTITIONED BY (FECHA STRING)
STORED AS AVRO
LOCATION '/user/proyecto/database/landing/transaccion'
TBLPROPERTIES (
'avro.schema.url'='hdfs:///user/proyecto/schema/landing/transaccion.avsc',
'avro.output.codec'='snappy'
);

In [ ]:
# Desde la consola Hive
hive> CREATE TABLE LANDING.TRANSACCION
    > PARTITIONED BY (FECHA STRING)
    > STORED AS AVRO
    > LOCATION '/user/proyecto/database/landing/transaccion'
    > TBLPROPERTIES (
    > 'avro.schema.url'='hdfs:///user/proyecto/schema/landing/transaccion.avsc',
    > 'avro.output.codec'='snappy'
    > );
OK
Time taken: 0.081 seconds
hive>

In [ ]:
------------------------------
INSERCION DE DATOS EN LA TABLA
------------------------------

-- Desde HIVE, ejecutamos la inserción de datos desde la tabla "LANDING_TMP" a la tabla "LANDING"
INSERT OVERWRITE TABLE LANDING.TRANSACCION
PARTITION(FECHA)
SELECT * FROM  LANDING_TMP.TRANSACCION;

In [ ]:
# Desde la consola Hive
hive> INSERT OVERWRITE TABLE LANDING.TRANSACCION
    > PARTITION(FECHA)
    > SELECT * FROM  LANDING_TMP.TRANSACCION;
WARNING: Hive-on-MR is deprecated in Hive 2 and may not be available in the future versions. Consider using a different execution engine (i.e. spark, tez) or using Hive 1.X releases.
Query ID = root_20240529181533_caca06e5-0164-44ab-a937-1ff2793df9ee
Total jobs = 3
Launching Job 1 out of 3
Number of reduce tasks is set to 0 since there's no reduce operator
Job running in-process (local Hadoop)
2024-05-29 18:15:34,570 Stage-1 map = 0%,  reduce = 0%
2024-05-29 18:15:35,580 Stage-1 map = 100%,  reduce = 0%
Ended Job = job_local473354807_0003
Stage-4 is selected by condition resolver.
Stage-3 is filtered out by condition resolver.
Stage-5 is filtered out by condition resolver.
Moving data to directory hdfs://namenode:8020/user/proyecto/database/landing/transaccion/.hive-staging_hive_2024-05-29_18-15-33_050_78981836514765872-1/-ext-10000
Loading data to table landing.transaccion partition (fecha=null)

Loaded : 4/4 partitions.
         Time taken to load dynamic partitions: 0.641 seconds
         Time taken for adding to write entity : 0.009 seconds
MapReduce Jobs Launched:
Stage-Stage-1:  HDFS Read: 5170849 HDFS Write: 1863365 SUCCESS
Total MapReduce CPU Time Spent: 0 msec
OK
Time taken: 3.52 seconds
hive>

In [ ]:
# Esta vez, como el particionado dinámico es true, Hive comprobará los valores de la columna de particionamiento y creará 
# una partición separada para cada valor único de esa columna:
root@0ab30c74254e:/home/hadoop/archivos# hdfs dfs -ls /user/proyecto/database/landing/transaccion
Found 4 items
drwxr-xr-x   - root supergroup          0 2024-05-29 18:15 /user/proyecto/database/landing/transaccion/fecha=2018-01-21
drwxr-xr-x   - root supergroup          0 2024-05-29 18:15 /user/proyecto/database/landing/transaccion/fecha=2018-01-22
drwxr-xr-x   - root supergroup          0 2024-05-29 18:15 /user/proyecto/database/landing/transaccion/fecha=2018-01-23
drwxr-xr-x   - root supergroup          0 2024-05-29 18:15 /user/proyecto/database/landing/transaccion/fecha=FECHA
root@0ab30c74254e:/home/hadoop/archivos#

In [ ]:
-- Desde HIVE, verificamos que se hayan insertado los datos en la tabla "LANDING"
SELECT * FROM LANDING.TRANSACCION LIMIT 10;

In [ ]:
hive> SELECT * FROM LANDING.TRANSACCION LIMIT 10;
OK
+-------------------------+-------------------------+--------------------+--------------------+
| transaccion.id_persona  | transaccion.id_empresa  | transaccion.monto  | transaccion.fecha  |
+-------------------------+-------------------------+--------------------+--------------------+
| 18                      | 3                       | 1383               | 2018-01-21         |
| 30                      | 6                       | 2331               | 2018-01-21         |
| 47                      | 2                       | 2280               | 2018-01-21         |
| 28                      | 1                       | 730                | 2018-01-21         |
| 91                      | 4                       | 3081               | 2018-01-21         |
| 74                      | 8                       | 2409               | 2018-01-21         |
| 83                      | 5                       | 2079               | 2018-01-21         |
| 48                      | 4                       | 2543               | 2018-01-21         |
| 15                      | 6                       | 1434               | 2018-01-21         |
| 89                      | 4                       | 780                | 2018-01-21         |
+-------------------------+-------------------------+--------------------+--------------------+

In [ ]:
-- Verificamos
SHOW PARTITIONS LANDING.TRANSACCION;
--   ________________
--  |   partition    |
--  |----------------|
--  |fecha=2018-01-21|
--  |fecha=2018-01-22|
--  |fecha=2018-01-23|
--  |fecha=FECHA     | <---------------- Debemos borrar esta particion erronea, eso lo
--  |________________|                   haremos en la capa UNIVERSAL.


In [ ]:
# Desde la consola Hive
hive> SHOW PARTITIONS LANDING.TRANSACCION;
OK
fecha=2018-01-21
fecha=2018-01-22
fecha=2018-01-23
fecha=FECHA
Time taken: 0.086 seconds, Fetched: 4 row(s)
hive>

#### **`Paso 5` - Crear la capa UNIVERSAL**

En este paso, los datos que han sido capturados y almacenados en la capa LANDING son transformados y luego cargados en la capa UNIVERSAL del Data Lake. La transformación puede incluir limpieza de datos, enriquecimiento con datos adicionales, integración con otros conjuntos de datos, y modelado de datos para prepararlos para su uso posterior. Este paso es crucial para garantizar que los datos estén en un formato estandarizado y listos para ser utilizados en múltiples casos de uso y aplicaciones.

Después de la transformación, los datos son cargados en la capa UNIVERSAL del Data Lake. Esta capa actúa como un repositorio centralizado de datos organizados y estructurados que pueden ser utilizados por diferentes equipos y aplicaciones en la organización. Es importante aplicar políticas de gobernanza de datos en esta etapa para garantizar la calidad, integridad y seguridad de los datos en la capa UNIVERSAL.

`Primer paso que realiza el Modelador`:
- Se realiza el Modelamiento de datos segun "Reglas de calidad" establecidas por el Cliente (o negocio). 
- En el modelamiento de datos como primer paso es limpiar los datos incorrectos y seleccionar cuales son los campos necesarios para poder construir los procesos que la empresa va a necesitar.
- Todas las tablas en esta capa se recomienda por estandar que esten en formato **PARQUET**.

`Segundo paso que realiza el Modelador`:
- Definir aquellas tablas intermedias que van a ser usadas por las soluciones en la capa `SMART`. Tabla intermedia se refiere a crear una tabla con registros de varias tablas para evitar que la capa `SMART` genere un gasto de recurso innecesario generando JOINS de tablas.

##### `PERSONA`

In [ ]:
                                              ------------
                                                 PERSONA    
                                              ------------
------------------
ACTIVAR COMPRESION
------------------

-- Desde HIVE, activamos la compresión y el formato SNAPPY (Ya lo hicimos durante la creación de la capa LANDING)
SET hive.exec.compress.output=true;
SET parquet.compression=SNAPPY;

In [ ]:
-----------
CREAR TABLA
-----------

-- Desde HIVE, creamos la tabla PERSONA (Ahora utilizamos los tipos de datos correctos)
CREATE TABLE UNIVERSAL.PERSONA(
ID STRING,
NOMBRE STRING,
TELEFONO STRING,
CORREO STRING,
FECHA_INGRESO STRING,
EDAD INT,
SALARIO DOUBLE,
ID_EMPRESA STRING
)
STORED AS PARQUET
LOCATION '/user/proyecto/database/universal/persona'
TBLPROPERTIES ("parquet.compression"="SNAPPY");

In [ ]:
# Desde la consola Hive
hive> CREATE TABLE UNIVERSAL.PERSONA(
    > ID STRING,
    > NOMBRE STRING,
    > TELEFONO STRING,
    > CORREO STRING,
    > FECHA_INGRESO STRING,
    > EDAD INT,
    > SALARIO DOUBLE,
    > ID_EMPRESA STRING
    > )
    > STORED AS PARQUET
    > LOCATION '/user/proyecto/database/universal/persona'
    > TBLPROPERTIES ("parquet.compression"="SNAPPY");
OK
Time taken: 0.1 seconds
hive>

In [ ]:
------------------------------
INSERCION DE DATOS EN LA TABLA
------------------------------

-- Desde HIVE, ejecutamos la inserción de datos desde la tabla "LANDING" a la tabla "UNIVERSAL"
INSERT OVERWRITE TABLE UNIVERSAL.PERSONA
SELECT
  cast(ID AS STRING),
  cast(NOMBRE AS STRING),
  cast(TELEFONO AS STRING),
  cast(CORREO AS STRING),
  cast(FECHA_INGRESO AS STRING),
  cast(EDAD AS INT),
  cast(SALARIO AS DOUBLE),
  cast(ID_EMPRESA AS STRING)
FROM 
  LANDING.PERSONA
WHERE 
  ID != 'ID';    <-------------------------- Con esta linea filtramos ese registro que nos mostraba
                                             los encabezados como un registro más.
 EDAD > 0        <-------------------------- Es aqui donde podemos comenzar a restringir los datos
 SALARIO > 1000                              (Estas restricciones son solo para ejemplo, no tomar en cuenta)

In [ ]:
# Desde consola Hive
hive> INSERT OVERWRITE TABLE UNIVERSAL.PERSONA
    > SELECT
    >   cast(ID AS STRING),
    >   cast(NOMBRE AS STRING),
    >   cast(TELEFONO AS STRING),
    >   cast(CORREO AS STRING),
    >   cast(FECHA_INGRESO AS STRING),
    >   cast(EDAD AS INT),
    >   cast(SALARIO AS DOUBLE),
    >   cast(ID_EMPRESA AS STRING)
    > FROM
    >   LANDING.PERSONA
    > WHERE
    >   ID != 'ID';
WARNING: Hive-on-MR is deprecated in Hive 2 and may not be available in the future versions. Consider using a different execution engine (i.e. spark, tez) or using Hive 1.X releases.
Query ID = root_20240529182237_a59d61c1-9020-4bef-8116-41c1cecf0a01
Total jobs = 3
Launching Job 1 out of 3
Number of reduce tasks is set to 0 since there's no reduce operator
Job running in-process (local Hadoop)
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
2024-05-29 18:22:39,292 Stage-1 map = 0%,  reduce = 0%
2024-05-29 18:22:40,298 Stage-1 map = 100%,  reduce = 0%
Ended Job = job_local1511578204_0004
Stage-4 is selected by condition resolver.
Stage-3 is filtered out by condition resolver.
Stage-5 is filtered out by condition resolver.
Moving data to directory hdfs://namenode:8020/user/proyecto/database/universal/persona/.hive-staging_hive_2024-05-29_18-22-37_242_7135171591015813583-1/-ext-10000
Loading data to table universal.persona
MapReduce Jobs Launched:
Stage-Stage-1:  HDFS Read: 5237277 HDFS Write: 1870618 SUCCESS
Total MapReduce CPU Time Spent: 0 msec
OK
Time taken: 3.467 seconds
hive>

In [ ]:
-- Desde HIVE, verificamos que se hayan insertado los datos en la tabla "LANDING"
SELECT * FROM UNIVERSAL.PERSONA LIMIT 10;

In [ ]:
hive> SELECT * FROM UNIVERSAL.PERSONA LIMIT 10;
OK
+-------------+-----------------+-------------------+-----------------------------------------+------------------------+---------------+------------------+---------------------+
| persona.id  | persona.nombre  | persona.telefono  |             persona.correo              | persona.fecha_ingreso  | persona.edad  | persona.salario  | persona.id_empresa  |
+-------------+-----------------+-------------------+-----------------------------------------+------------------------+---------------+------------------+---------------------+
| 1           | Carl            | 1-745-633-9145    | arcu.Sed.et@ante.co.uk                  | 2004-04-23             | 32            | 20095.0          | 5                   |
| 2           | Priscilla       | 155-2498          | Donec.egestas.Aliquam@volutpatnunc.edu  | 2019-02-17             | 34            | 9298.0           | 2                   |
| 3           | Jocelyn         | 1-204-956-8594    | amet.diam@lobortis.co.uk                | 2002-08-01             | 27            | 10853.0          | 3                   |
| 4           | Aidan           | 1-719-862-9385    | euismod.et.commodo@nibhlaciniaorci.edu  | 2018-11-06             | 29            | 3387.0           | 10                  |
| 5           | Leandra         | 839-8044          | at@pretiumetrutrum.com                  | 2002-10-10             | 41            | 22102.0          | 1                   |
| 6           | Bert            | 797-4453          | a.felis.ullamcorper@arcu.org            | 2017-04-25             | 70            | 7800.0           | 7                   |
| 7           | Mark            | 1-680-102-6792    | Quisque.ac@placerat.ca                  | 2006-04-21             | 52            | 8112.0           | 5                   |
| 8           | Jonah           | 214-2975          | eu.ultrices.sit@vitae.ca                | 2017-10-07             | 23            | 17040.0          | 5                   |
| 9           | Hanae           | 935-2277          | eu@Nunc.ca                              | 2003-05-25             | 69            | 6834.0           | 3                   |
| 10          | Cadman          | 1-866-561-2701    | orci.adipiscing.non@semperNam.ca        | 2001-05-19             | 19            | 7996.0           | 7                   |
+-------------+-----------------+-------------------+-----------------------------------------+------------------------+---------------+------------------+---------------------+

In [ ]:
-- Verificamos los tipos de datos de sus campos
DESCRIBE UNIVERSAL.PERSONA;

In [ ]:
hive> DESCRIBE UNIVERSAL.PERSONA;
OK
+----------------+------------+----------+
|    col_name    | data_type  | comment  |
+----------------+------------+----------+
| id             | string     |          |
| nombre         | string     |          |
| telefono       | string     |          |
| correo         | string     |          |
| fecha_ingreso  | string     |          |
| edad           | int        |          |
| salario        | double     |          |
| id_empresa     | string     |          |
+----------------+------------+----------+
Time taken: 0.047 seconds, Fetched: 8 row(s)
hive>

##### `EMPRESA`

In [ ]:
                                              ------------
                                                 EMPRESA    
                                              ------------
------------------
ACTIVAR COMPRESION
------------------

-- Desde HIVE, activamos la compresión y el formato SNAPPY (Ya lo hicimos durante la creación de la capa LANDING)
SET hive.exec.compress.output=true;
SET parquet.compression=SNAPPY;

In [ ]:
-----------
CREAR TABLA
-----------

-- Desde HIVE, creamos la tabla EMPRESA (Ahora utilizamos los tipos de datos correctos)
CREATE TABLE UNIVERSAL.EMPRESA(
ID STRING,
NOMBRE STRING
)
STORED AS PARQUET
LOCATION '/user/proyecto/database/universal/empresa'
TBLPROPERTIES ("parquet.compression"="SNAPPY");

In [ ]:
# Desde la consola Hive
hive> CREATE TABLE UNIVERSAL.EMPRESA(
    > ID STRING,
    > NOMBRE STRING
    > )
    > STORED AS PARQUET
    > LOCATION '/user/proyecto/database/universal/empresa'
    > TBLPROPERTIES ("parquet.compression"="SNAPPY");
OK
Time taken: 0.056 seconds
hive>

In [ ]:
------------------------------
INSERCION DE DATOS EN LA TABLA
------------------------------

-- Desde HIVE, ejecutamos la inserción de datos desde la tabla "LANDING" a la tabla "UNIVERSAL"
INSERT OVERWRITE TABLE UNIVERSAL.EMPRESA
SELECT
  cast(ID AS STRING),
  cast(NOMBRE AS STRING)
FROM 
  LANDING.EMPRESA
WHERE 
  ID != 'ID';    <-------------------------- Con esta linea filtramos ese registro que nos mostraba
                                             los encabezados como un registro más.

In [ ]:
hive> INSERT OVERWRITE TABLE UNIVERSAL.EMPRESA
    > SELECT
    >   cast(ID AS STRING),
    >   cast(NOMBRE AS STRING)
    > FROM
    >   LANDING.EMPRESA
    > WHERE
    >   ID != 'ID';
WARNING: Hive-on-MR is deprecated in Hive 2 and may not be available in the future versions. Consider using a different execution engine (i.e. spark, tez) or using Hive 1.X releases.
Query ID = root_20240529182622_b96f210e-f8fc-495b-a2d5-07a33effb4e3
Total jobs = 3
Launching Job 1 out of 3
Number of reduce tasks is set to 0 since there's no reduce operator
Job running in-process (local Hadoop)
2024-05-29 18:26:24,120 Stage-1 map = 100%,  reduce = 0%
Ended Job = job_local733136724_0005
Stage-4 is selected by condition resolver.
Stage-3 is filtered out by condition resolver.
Stage-5 is filtered out by condition resolver.
Moving data to directory hdfs://namenode:8020/user/proyecto/database/universal/empresa/.hive-staging_hive_2024-05-29_18-26-22_467_6398841717238743219-1/-ext-10000
Loading data to table universal.empresa
MapReduce Jobs Launched:
Stage-Stage-1:  HDFS Read: 5247024 HDFS Write: 1871151 SUCCESS
Total MapReduce CPU Time Spent: 0 msec
OK
Time taken: 2.097 seconds
hive>

In [ ]:
-- Desde HIVE, verificamos que se hayan insertado los datos en la tabla "LANDING"
SELECT * FROM UNIVERSAL.EMPRESA LIMIT 10;

In [ ]:
hive> SELECT * FROM UNIVERSAL.EMPRESA LIMIT 10;
OK
+-------------+-----------------+
| empresa.id  | empresa.nombre  |
+-------------+-----------------+
| 1           | Walmart         |
| 2           | Microsoft       |
| 3           | Apple           |
| 4           | Toyota          |
| 5           | Amazon          |
| 6           | Google          |
| 7           | Samsung         |
| 8           | HP              |
| 9           | IBM             |
| 10          | Sony            |
+-------------+-----------------+
Time taken: 0.153 seconds, Fetched: 10 row(s)
hive>

In [ ]:
-- Verificamos los tipos de datos de sus campos
DESCRIBE UNIVERSAL.EMPRESA;

In [ ]:
hive> DESCRIBE UNIVERSAL.EMPRESA;
OK
+-----------+------------+----------+
| col_name  | data_type  | comment  |
+-----------+------------+----------+
| id        | string     |          |
| nombre    | string     |          |
+-----------+------------+----------+
Time taken: 0.024 seconds, Fetched: 2 row(s)
hive>

##### `TRANSACCION`

In [ ]:
                                            ----------------
                                               TRANSACCION    
                                            ----------------
------------------
ACTIVAR COMPRESION 
------------------

-- Desde HIVE, activamos la compresión y el formato SNAPPY (Ya lo hicimos durante la creación de la capa LANDING)
SET hive.exec.compress.output=true;
SET parquet.compression=SNAPPY;

In [ ]:
---------------------------------
ACTIVAR PARTICIONAMIENTO DINAMICO
---------------------------------

-- Activamos el particionamiento dinámico (Ya lo hicimos durante la creación de la capa LANDING)

SET hive.exec.dynamic.partition=true;
SET hive.exec.dynamic.partition.mode=nonstrict;

In [ ]:
-----------
CREAR TABLA
-----------

-- Desde HIVE, creamos la tabla TRANSACCION (Ahora utilizamos los tipos de datos correctos)
CREATE TABLE UNIVERSAL.TRANSACCION(
ID_PERSONA STRING,
ID_EMPRESA STRING,
MONTO DOUBLE
)
PARTITIONED BY (FECHA STRING)
STORED AS PARQUET
LOCATION '/user/proyecto/database/universal/transaccion'
TBLPROPERTIES ("parquet.compression"="SNAPPY");

In [ ]:
# Desde la consola Hive
hive> CREATE TABLE UNIVERSAL.TRANSACCION(
    > ID_PERSONA STRING,
    > ID_EMPRESA STRING,
    > MONTO DOUBLE
    > )
    > PARTITIONED BY (FECHA STRING)
    > STORED AS PARQUET
    > LOCATION '/user/proyecto/database/universal/transaccion'
    > TBLPROPERTIES ("parquet.compression"="SNAPPY");
OK
Time taken: 0.055 seconds
hive>

In [ ]:
------------------------------
INSERCION DE DATOS EN LA TABLA
------------------------------

-- Desde HIVE, ejecutamos la inserción de datos desde la tabla "LANDING" a la tabla "UNIVERSAL"
INSERT OVERWRITE TABLE UNIVERSAL.TRANSACCION
PARTITION(FECHA)
SELECT
  cast(ID_PERSONA AS STRING),
  cast(ID_EMPRESA AS STRING),
  cast(MONTO AS DOUBLE),
  cast(FECHA AS STRING)
FROM 
  LANDING.TRANSACCION
WHERE 
  ID_PERSONA != 'ID_PERSONA';    <-------------------------- Con esta linea filtramos ese registro que nos mostraba
                                                             los encabezados como un registro más.

In [ ]:
# Desde consola Hive
hive> INSERT OVERWRITE TABLE UNIVERSAL.TRANSACCION
    > PARTITION(FECHA)
    > SELECT
    >   cast(ID_PERSONA AS STRING),
    >   cast(ID_EMPRESA AS STRING),
    >   cast(MONTO AS DOUBLE),
    >   cast(FECHA AS STRING)
    > FROM
    >   LANDING.TRANSACCION
    > WHERE
    >   ID_PERSONA != 'ID_PERSONA';
WARNING: Hive-on-MR is deprecated in Hive 2 and may not be available in the future versions. Consider using a different execution engine (i.e. spark, tez) or using Hive 1.X releases.
Query ID = root_20240529183950_ffe6d383-d10f-4dbf-8050-f53dff0e4842
Total jobs = 3
Launching Job 1 out of 3
Number of reduce tasks is set to 0 since there's no reduce operator
Job running in-process (local Hadoop)
2024-05-29 18:39:51,852 Stage-1 map = 0%,  reduce = 0%
2024-05-29 18:39:53,863 Stage-1 map = 100%,  reduce = 0%
Ended Job = job_local289072163_0006
Stage-4 is selected by condition resolver.
Stage-3 is filtered out by condition resolver.
Stage-5 is filtered out by condition resolver.
Moving data to directory hdfs://namenode:8020/user/proyecto/database/universal/transaccion/.hive-staging_hive_2024-05-29_18-39-50_172_2568936396880136650-1/-ext-10000
Loading data to table universal.transaccion partition (fecha=null)

Loaded : 3/3 partitions.
         Time taken to load dynamic partitions: 0.462 seconds
         Time taken for adding to write entity : 0.0 seconds
MapReduce Jobs Launched:
Stage-Stage-1:  HDFS Read: 7132871 HDFS Write: 2623049 SUCCESS
Total MapReduce CPU Time Spent: 0 msec
OK
Time taken: 4.609 seconds
hive>

In [ ]:
# Esta vez, como el particionado dinámico es true, Hive comprobará los valores de la columna de particionamiento y creará 
# una partición separada para cada valor único de esa columna:
root@0ab30c74254e:/home/hadoop/archivos# hdfs dfs -ls /user/proyecto/database/universal/transaccion
Found 3 items
drwxr-xr-x   - root supergroup          0 2024-05-29 18:39 /user/proyecto/database/universal/transaccion/fecha=2018-01-21
drwxr-xr-x   - root supergroup          0 2024-05-29 18:39 /user/proyecto/database/universal/transaccion/fecha=2018-01-22
drwxr-xr-x   - root supergroup          0 2024-05-29 18:39 /user/proyecto/database/universal/transaccion/fecha=2018-01-23
root@0ab30c74254e:/home/hadoop/archivos#

In [ ]:
-- Desde HIVE, verificamos que se hayan insertado los datos en la tabla "LANDING"
SELECT * FROM UNIVERSAL.TRANSACCION LIMIT 10;

In [ ]:
hive> SELECT * FROM UNIVERSAL.TRANSACCION LIMIT 10;
OK
+-------------------------+-------------------------+--------------------+--------------------+
| transaccion.id_persona  | transaccion.id_empresa  | transaccion.monto  | transaccion.fecha  |
+-------------------------+-------------------------+--------------------+--------------------+
| 18                      | 3                       | 1383.0             | 2018-01-21         |
| 30                      | 6                       | 2331.0             | 2018-01-21         |
| 47                      | 2                       | 2280.0             | 2018-01-21         |
| 28                      | 1                       | 730.0              | 2018-01-21         |
| 91                      | 4                       | 3081.0             | 2018-01-21         |
| 74                      | 8                       | 2409.0             | 2018-01-21         |
| 83                      | 5                       | 2079.0             | 2018-01-21         |
| 48                      | 4                       | 2543.0             | 2018-01-21         |
| 15                      | 6                       | 1434.0             | 2018-01-21         |
| 89                      | 4                       | 780.0              | 2018-01-21         |
+-------------------------+-------------------------+--------------------+--------------------+
Time taken: 0.165 seconds, Fetched: 10 row(s)
hive>

In [ ]:
-- Verificamos
SHOW PARTITIONS UNIVERSAL.TRANSACCION;
--   ________________
--  |   partition    |
--  |----------------|
--  |fecha=2018-01-21|
--  |fecha=2018-01-22|
--  |fecha=2018-01-23|
--  |________________|

In [ ]:
hive> SHOW PARTITIONS UNIVERSAL.TRANSACCION;
OK
fecha=2018-01-21
fecha=2018-01-22
fecha=2018-01-23
Time taken: 0.048 seconds, Fetched: 3 row(s)
hive>

In [ ]:
-- Verificamos los tipos de datos de sus campos
DESCRIBE UNIVERSAL.TRANSACCION;

In [ ]:
hive> DESCRIBE UNIVERSAL.TRANSACCION;
OK
+--------------------------+------------+----------+
|         col_name         | data_type  | comment  |
+--------------------------+------------+----------+
| id_persona               | string     |          |
| id_empresa               | string     |          |
| monto                    | double     |          |
| fecha                    | string     |          |
|                          | NULL       | NULL     |
| # Partition Information  | NULL       | NULL     |
| # col_name               | data_type  | comment  |
| fecha                    | string     |          |
+--------------------------+------------+----------+
Time taken: 0.066 seconds, Fetched: 9 row(s)
hive>

##### `TRANSACCION_ENRIQUECIDA`

In [ ]:
                                              ---------------------------
                                                TRANSACCION_ENRIQUECIDA
                                              ---------------------------
------------------
ACTIVAR COMPRESION 
------------------

-- Desde HIVE, activamos la compresión y el formato SNAPPY (Ya lo hicimos durante la creación de la capa LANDING)
SET hive.exec.compress.output=true;
SET parquet.compression=SNAPPY;

In [ ]:
---------------------------------
ACTIVAR PARTICIONAMIENTO DINAMICO
---------------------------------

-- Activamos el particionamiento dinámico (Ya lo hicimos durante la creación de la capa LANDING)

SET hive.exec.dynamic.partition=true;
SET hive.exec.dynamic.partition.mode=nonstrict;

In [ ]:
-----------
CREAR TABLA
-----------

-- Desde HIVE, creamos la tabla TRANSACCION_ENRIQUECIDA
CREATE TABLE UNIVERSAL.TRANSACCION_ENRIQUECIDA(
ID_PERSONA INT,
NOMBRE_PERSONA STRING,
EDAD_PERSONA INT,
SALARIO_PERSONA DOUBLE,
TRABAJO_PERSONA STRING,
MONTO_TRANSACCION DOUBLE,
EMPRESA_TRANSACCION STRING
)
PARTITIONED BY (FECHA_TRANSACCION STRING)
STORED AS PARQUET
LOCATION '/user/proyecto/database/universal/transaccion_enriquecida'
TBLPROPERTIES ("parquet.compression"="SNAPPY");

In [ ]:
# Desde la consola Hive
hive> CREATE TABLE UNIVERSAL.TRANSACCION_ENRIQUECIDA(
    > ID_PERSONA INT,
    > NOMBRE_PERSONA STRING,
    > EDAD_PERSONA INT,
    > SALARIO_PERSONA DOUBLE,
    > TRABAJO_PERSONA STRING,
    > MONTO_TRANSACCION DOUBLE,
    > EMPRESA_TRANSACCION STRING
    > )
    > PARTITIONED BY (FECHA_TRANSACCION STRING)
    > STORED AS PARQUET
    > LOCATION '/user/proyecto/database/universal/transaccion_enriquecida'
    > TBLPROPERTIES ("parquet.compression"="SNAPPY");
OK
Time taken: 0.097 seconds
hive>

#### **`Paso 6` - Crear una tabla desnormalizada sobre la capa UNIVERSAL**

##### **Estructura, descripción y tipos de datos de la tabla `TRANSACCION_ENRIQUECIDA`**

```text
En la capa "universal" se creó la tabla "transaccion_enriquecida" la cual debe ser construida en función de las 
tablas "persona", "empresa" y "transaccion". Los campos de la tabla son los siguientes:

               _____________________________________________________________________________________
              |        CAMPO        |  TIPO  |                    DESCRIPCION                       |
              |---------------------|--------|------------------------------------------------------|
              | ID_PERSONA          | INT    | ID de la persona que realizo la transaccion          |
              |---------------------|--------|------------------------------------------------------|
              | NOMBRE_PERSONA      | STRING | Nombre de la persona que realizó la transaccion      |
              |---------------------|--------|------------------------------------------------------|
              | EDAD_PERSONA        | INT    | Edad de la persona que realizó la transaccion        |
              |---------------------|--------|------------------------------------------------------|
              | SALARIO_PERSONA     | DOUBLE | Salario de la persona que realizó la transaccion     |
              |---------------------|--------|------------------------------------------------------|
              | TRABAJO_PERSONA     | STRING | Nombre de la empresa donde trabaja la persona        |
              |---------------------|--------|------------------------------------------------------|
              | MONTO_TRANSACCION   | DOUBLE | Monto de la transaccion realizada                    |
              |---------------------|--------|------------------------------------------------------|
              | FECHA_TRANSACCION   | STRING | Fecha de la transaccion realizada                    |
              |---------------------|--------|------------------------------------------------------|
              | EMPRESA_TRANSACCION | STRING | Nombre de la empresa donde se realizó la transaccion |
              |_____________________|________|______________________________________________________|
```             

##### **Diagrama de proceso para la obtención de la tabla `TRANSACCION_ENRIQUECIDA`**

In [ ]:
----------------------------------------------
DIAGRAMA INPUT/INTERMEDIATE/OUTPUT [PROCESO 1]
----------------------------------------------

TRANSACCION_PERSONA_ENRIQUECIDA_1         -----.
TRANSACCION_PERSONA_ENRIQUECIDA_2              |------> TABLAS TEMPORALES 
TRANSACCION_PERSONA_EMPRESA_ENRIQUECIDA   -----'    Seran creadas como tablas temporales
                                                    (al cerrar sesion se borran automaticamente)
                                                    porque solo nos sirven para obtener la     
                                                    tabla output "TRANSACCION_ENRIQUECIDA"

                                                                                                    CAPA UNIVERSAL                                                    
                                                                                                    ==============

TRANSACCION                                 TRANSACCION_PERSONA_ENRIQUECIDA_1
 ________                                              ________    ID_PERSONA
|__|__|__|                                            |__|__|__|   NOMBRE_PERSONA
|__|__|__|--------.                        .--------> |__|__|__|   EDAD_PERSONA
|__|__|__|        |                        |          |__|__|__|   SALARIO_PERSONA
                  |         ______         |               |       ID_EMPRESA_PERSONA ----> Hace referencia al ID de la Empresa donde la Persona trabaja 
PERSONA           |--------| JOIN |--------'               |       MONTO_TRANSACCION
 ________         |        |______|                        |       FECHA_TRANSACCION 
|__|__|__|        |   TRANSACCION.ID_PERSONA               |       ID_EMPRESA_TRANSACCION ----> Hace referencia al ID de la Empresa |donde se realizo la transaccion
|__|__|__|--------'             =                          | 
|__|__|__|                  PERSONA.ID                     | 
                                                           |                                                                                                                                                TRANSACCION_ENRIQUECIDA                                                  
EMPRESA                                                    |                         TRANSACCION_PERSONA_ENRIQUECIDA_2                                                                                             ________ 
 ________                                                __˅___                                  ________  ID_PERSONA                                                                       ________              |__|__|__|  
|__|__|__|                                              | JOIN |                                |__|__|__| NOMBRE_PERSONA                                    .---------------------------> | INSERT |-----------> |__|__|__|
|__|__|__|--------------------------------------------->|______|------------------------------> |__|__|__| EDAD_PERSONA                                      |                             |________|             |__|__|__|
|__|__|__|                             TRANSACCION_PERSONA_ENRIQUECIDA_1.ID_EMPRESA             |__|__|__| SALARIO_PERSONA                                   | 
    |                                                       =                                       |      TRABAJO_PERSONA                                   |   
    |                                                   EMPRESA.ID                                  |      MONTO_TRANSACCION                                 | 
    |                                                                                               |      FECHA_TRANSACCION                                 |
    |                                                                                               |      ID_EMPRESA_TRANSACCION       TRANSACCION_PERSONA_EMPRESA_ENRIQUECIDA
    |                                                                                               |                                                    ____|___  ID_PERSONA
    |                                                                                            ___˅__                                                 |__|__|__| NOMBRE_PERSONA
--  '-----------------------------------------------------------------------------------------> | JOIN |----------------------------------------------> |__|__|__| EDAD_PERSONA
                                                                                                |______|                                                |__|__|__| SALARIO_PERSONA
                                                                      TRANSACCION_PERSONA_ENRIQUECIDA_2.ID_EMPRESA_TRANSACCION                                     TRABAJO_PERSONA 
                                                                                                   =                                                               MONTO_TRANSACCION 
                                                                                               EMPRESA.ID                                                          FECHA_TRANSACCION 
                                                                                                                                                                   EMPRESA_TRANSACCION 
 

##### **Truncar la tabla `TRANSACCION_ENRIQUECIDA`**

In [ ]:
TRUNCATE TABLE UNIVERSAL.TRANSACCION_ENRIQUECIDA; <----------- Truncamos la tabla para asegurarnos de que este vacia.                                                                    

In [ ]:
# Desde la consola Hive
hive> TRUNCATE TABLE UNIVERSAL.TRANSACCION_ENRIQUECIDA;
OK
Time taken: 0.139 seconds
hive>

##### **Creación e inserción de datos para la tabla temporal `TRANSACCION_PERSONA_ENRIQUECIDA_1`**

In [ ]:
DROP TABLE IF EXISTS UNIVERSAL.TRANSACCION_PERSONA_ENRIQUECIDA_1;
CREATE TEMPORARY TABLE UNIVERSAL.TRANSACCION_PERSONA_ENRIQUECIDA_1(
ID_PERSONA STRING,
NOMBRE_PERSONA STRING,
EDAD_PERSONA INT,                                -- No es necesario COMPRIMIR una Tabla temporal
SALARIO_PERSONA DOUBLE,                          -- Como tampoco indicar el LOCATION
ID_EMPRESA_PERSONA STRING,
MONTO_TRANSACCION DOUBLE,
FECHA_TRANSACCION STRING,
ID_EMPRESA_TRANSACCION STRING
)
STORED AS PARQUET;

In [ ]:
# Desde la consola Hive
hive> DROP TABLE IF EXISTS UNIVERSAL.TRANSACCION_PERSONA_ENRIQUECIDA_1;
OK
Time taken: 0.91 seconds
hive> CREATE TEMPORARY TABLE UNIVERSAL.TRANSACCION_PERSONA_ENRIQUECIDA_1(
    > ID_PERSONA STRING,
    > NOMBRE_PERSONA STRING,
    > EDAD_PERSONA INT,
    > SALARIO_PERSONA DOUBLE,
    > ID_EMPRESA_PERSONA STRING,
    > MONTO_TRANSACCION DOUBLE,
    > FECHA_TRANSACCION STRING,
    > ID_EMPRESA_TRANSACCION STRING
    > )
    > STORED AS PARQUET;
OK
Time taken: 0.311 seconds
hive>

In [ ]:
-- Copiamos datos del join entre las tablas "UNIVERSAL.TRANSACCION" y "UNIVERSAL.PERSONA" a la tabla "UNIVERSAL.TRANSACCION_PERSONA_ENRIQUECIDA_1"
INSERT INTO UNIVERSAL.TRANSACCION_PERSONA_ENRIQUECIDA_1
SELECT
  T.ID_PERSONA AS ID_PERSONA,
  P.NOMBRE AS NOMBRE_PERSONA,
  P.EDAD AS EDAD_PERSONA,
  P.SALARIO AS SALARIO_PERSONA,
  P.ID_EMPRESA AS ID_EMPRESA_PERSONA,
  T.MONTO AS MONTO_TRANSACCION,
  T.FECHA AS FECHA_TRANSACCION,
  T.ID_EMPRESA AS ID_EMPRESA_TRANSACCION
FROM
  UNIVERSAL.TRANSACCION T
JOIN 
  UNIVERSAL.PERSONA P
ON 
  T.ID_PERSONA = P.ID;

In [ ]:
# Desde la consola Hive
hive> INSERT INTO UNIVERSAL.TRANSACCION_PERSONA_ENRIQUECIDA_1
    > SELECT
    >   T.ID_PERSONA AS ID_PERSONA,
    >   P.NOMBRE AS NOMBRE_PERSONA,
    >   P.EDAD AS EDAD_PERSONA,
    >   P.SALARIO AS SALARIO_PERSONA,
    >   P.ID_EMPRESA AS ID_EMPRESA_PERSONA,
    >   T.MONTO AS MONTO_TRANSACCION,
    >   T.FECHA AS FECHA_TRANSACCION,
    >   T.ID_EMPRESA AS ID_EMPRESA_TRANSACCION
    > FROM
    >   UNIVERSAL.TRANSACCION T
    > JOIN
    >   UNIVERSAL.PERSONA P
    > ON
    >   T.ID_PERSONA = P.ID;
WARNING: Hive-on-MR is deprecated in Hive 2 and may not be available in the future versions. Consider using a different execution engine (i.e. spark, tez) or using Hive 1.X releases.
Query ID = root_20240529202927_5810fa1e-e43b-4a68-a226-5965d7d4e007
Total jobs = 1
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/opt/hive/lib/log4j-slf4j-impl-2.6.2.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/opt/hadoop-2.7.4/share/hadoop/common/lib/slf4j-log4j12-1.7.10.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Execution log at: /tmp/root/root_20240529202927_5810fa1e-e43b-4a68-a226-5965d7d4e007.log
2024-05-29 20:29:47     Starting to launch local task to process map join;      maximum memory = 477626368
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
2024-05-29 20:29:51     Dump the side-table for tag: 1 with group count: 100 into file: file:/tmp/root/b8f872cc-0994-46c8-a95b-abfe851a5b40/hive_2024-05-29_20-29-27_170_1433163670814785237-1/-local-10002/HashTable-Stage-4/MapJoin-mapfile01--.hashtable
2024-05-29 20:29:51     Uploaded 1 File to: file:/tmp/root/b8f872cc-0994-46c8-a95b-abfe851a5b40/hive_2024-05-29_20-29-27_170_1433163670814785237-1/-local-10002/HashTable-Stage-4/MapJoin-mapfile01--.hashtable (4122 bytes)
2024-05-29 20:29:51     End of local task; Time Taken: 3.501 sec.
Execution completed successfully
MapredLocal task succeeded
Launching Job 1 out of 1
Number of reduce tasks is set to 0 since there's no reduce operator
Job running in-process (local Hadoop)
2024-05-29 20:29:54,269 Stage-4 map = 0%,  reduce = 0%
2024-05-29 20:29:56,278 Stage-4 map = 100%,  reduce = 0%
Ended Job = job_local776073659_0007
Loading data to table universal.transaccion_persona_enriquecida_1
MapReduce Jobs Launched:
Stage-Stage-4:  HDFS Read: 8044534 HDFS Write: 4075645 SUCCESS
Total MapReduce CPU Time Spent: 0 msec
OK
Time taken: 29.24 seconds
hive>

In [ ]:
-- Desde HIVE, verificamos que se hayan insertado los datos en la tabla
SELECT * FROM UNIVERSAL.TRANSACCION_PERSONA_ENRIQUECIDA_1 LIMIT 10;

In [ ]:
hive> SELECT * FROM UNIVERSAL.TRANSACCION_PERSONA_ENRIQUECIDA_1 LIMIT 10;
OK
+-----------------------------------------------+---------------------------------------------------+-------------------------------------------------+----------------------------------------------------+------------------------------------------------------+-----------------------------------------------------+-----------------------------------------------------+----------------------------------------------------------+
| transaccion_persona_enriquecida_1.id_persona  | transaccion_persona_enriquecida_1.nombre_persona  | transaccion_persona_enriquecida_1.edad_persona  | transaccion_persona_enriquecida_1.salario_persona  | transaccion_persona_enriquecida_1.id_empresa_persona | transaccion_persona_enriquecida_1.monto_transaccion | transaccion_persona_enriquecida_1.fecha_transaccion | transaccion_persona_enriquecida_1.id_empresa_transaccion |
+-----------------------------------------------+---------------------------------------------------+-------------------------------------------------+----------------------------------------------------+------------------------------------------------------+-----------------------------------------------------+-----------------------------------------------------+----------------------------------------------------------+
| 18                                            | Owen                                              | 34                                              | 4759.0                                             | 7                                                    | 1383.0                                              | 2018-01-21                                          | 3                                                        |
| 30                                            | Clayton                                           | 52                                              | 9505.0                                             | 8                                                    | 2331.0                                              | 2018-01-21                                          | 6                                                        |
| 47                                            | Vernon                                            | 35                                              | 7109.0                                             | 10                                                   | 2280.0                                              | 2018-01-21                                          | 2                                                        |
| 28                                            | Stephen                                           | 53                                              | 9469.0                                             | 8                                                    | 730.0                                               | 2018-01-21                                          | 1                                                        |
| 91                                            | Erica                                             | 32                                              | 8934.0                                             | 6                                                    | 3081.0                                              | 2018-01-21                                          | 4                                                        |
| 74                                            | Kaitlin                                           | 56                                              | 6515.0                                             | 10                                                   | 2409.0                                              | 2018-01-21                                          | 8                                                        |
| 83                                            | Giselle                                           | 45                                              | 2503.0                                             | 2                                                    | 2079.0                                              | 2018-01-21                                          | 5                                                        |
| 48                                            | Illiana                                           | 18                                              | 1454.0                                             | 8                                                    | 2543.0                                              | 2018-01-21                                          | 4                                                        |
| 15                                            | Wanda                                             | 27                                              | 1539.0                                             | 5                                                    | 1434.0                                              | 2018-01-21                                          | 6                                                        |
| 89                                            | Kelly                                             | 55                                              | 10110.0                                            | 6                                                    | 780.0                                               | 2018-01-21                                          | 4                                                        |
+-----------------------------------------------+---------------------------------------------------+-------------------------------------------------+----------------------------------------------------+------------------------------------------------------+-----------------------------------------------------+-----------------------------------------------------+----------------------------------------------------------+
Time taken: 0.581 seconds, Fetched: 10 row(s)
hive>


In [ ]:
-- Verificamos los tipos de datos de sus campos
DESCRIBE UNIVERSAL.TRANSACCION_PERSONA_ENRIQUECIDA_1;

In [ ]:
hive> DESCRIBE UNIVERSAL.TRANSACCION_PERSONA_ENRIQUECIDA_1;
OK
+-------------------------+------------+----------+
|        col_name         | data_type  | comment  |
+-------------------------+------------+----------+
| id_persona              | string     |          |
| nombre_persona          | string     |          |
| edad_persona            | int        |          |
| salario_persona         | double     |          |
| id_empresa_persona      | string     |          |
| monto_transaccion       | double     |          |
| fecha_transaccion       | string     |          |
| id_empresa_transaccion  | string     |          |
+-------------------------+------------+----------+
Time taken: 0.029 seconds, Fetched: 8 row(s)
hive>


##### **Creación e inserción de datos para la tabla temporal `TRANSACCION_PERSONA_ENRIQUECIDA_2`**

In [ ]:
-- Eliminamos y volvemos a crear la tabla temporal
DROP TABLE IF EXISTS UNIVERSAL.TRANSACCION_PERSONA_ENRIQUECIDA_2;
CREATE TEMPORARY TABLE UNIVERSAL.TRANSACCION_PERSONA_ENRIQUECIDA_2(
ID_PERSONA STRING,
NOMBRE_PERSONA STRING,
EDAD_PERSONA INT,
SALARIO_PERSONA DOUBLE,
TRABAJO_PERSONA STRING,
MONTO_TRANSACCION DOUBLE,
FECHA_TRANSACCION STRING,
ID_EMPRESA_TRANSACCION STRING
)
STORED AS PARQUET;

In [ ]:
# Desde consola Hive
hive> DROP TABLE IF EXISTS UNIVERSAL.TRANSACCION_PERSONA_ENRIQUECIDA_2;
OK
Time taken: 0.099 seconds
hive> CREATE TEMPORARY TABLE UNIVERSAL.TRANSACCION_PERSONA_ENRIQUECIDA_2(
    > ID_PERSONA STRING,
    > NOMBRE_PERSONA STRING,
    > EDAD_PERSONA INT,
    > SALARIO_PERSONA DOUBLE,
    > TRABAJO_PERSONA STRING,
    > MONTO_TRANSACCION DOUBLE,
    > FECHA_TRANSACCION STRING,
    > ID_EMPRESA_TRANSACCION STRING
    > )
    > STORED AS PARQUET;
OK
Time taken: 0.066 seconds
hive>

In [ ]:
-- Copiamos datos del join entre las tablas "TRANSACCION_PERSONA_ENRIQUECIDA_1" y "UNIVERSAL.EMPRESA" a la tabla "UNIVERSAL.TRANSACCION_PERSONA_ENRIQUECIDA_2"
INSERT INTO UNIVERSAL.TRANSACCION_PERSONA_ENRIQUECIDA_2
SELECT
  T.ID_PERSONA AS ID_PERSONA,
  T.NOMBRE_PERSONA AS NOMBRE_PERSONA,
  T.EDAD_PERSONA AS EDAD_PERSONA,
  T.SALARIO_PERSONA AS SALARIO_PERSONA,
  E.NOMBRE AS TRABAJO_PERSONA,
  T.MONTO_TRANSACCION AS MONTO_TRANSACCION,
  T.FECHA_TRANSACCION AS FECHA_TRANSACCION,
  T.ID_EMPRESA_TRANSACCION AS ID_EMPRESA_TRANSACCION
FROM
  UNIVERSAL.TRANSACCION_PERSONA_ENRIQUECIDA_1 T
JOIN 
  UNIVERSAL.EMPRESA E
ON 
  T.ID_EMPRESA_PERSONA = E.ID;

In [ ]:
# Desde consola Hive
hive> INSERT INTO UNIVERSAL.TRANSACCION_PERSONA_ENRIQUECIDA_2
    > SELECT
    >   T.ID_PERSONA AS ID_PERSONA,
    >   T.NOMBRE_PERSONA AS NOMBRE_PERSONA,
    >   T.EDAD_PERSONA AS EDAD_PERSONA,
    >   T.SALARIO_PERSONA AS SALARIO_PERSONA,
    >   E.NOMBRE AS TRABAJO_PERSONA,
    >   T.MONTO_TRANSACCION AS MONTO_TRANSACCION,
    >   T.FECHA_TRANSACCION AS FECHA_TRANSACCION,
    >   T.ID_EMPRESA_TRANSACCION AS ID_EMPRESA_TRANSACCION
    > FROM
    >   UNIVERSAL.TRANSACCION_PERSONA_ENRIQUECIDA_1 T
    > JOIN
    >   UNIVERSAL.EMPRESA E
    > ON
    >   T.ID_EMPRESA_PERSONA = E.ID;
WARNING: Hive-on-MR is deprecated in Hive 2 and may not be available in the future versions. Consider using a different execution engine (i.e. spark, tez) or using Hive 1.X releases.
Query ID = root_20240529203505_1f0cd7e2-d35f-4777-9834-393e39c250c5
Total jobs = 1
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/opt/hive/lib/log4j-slf4j-impl-2.6.2.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/opt/hadoop-2.7.4/share/hadoop/common/lib/slf4j-log4j12-1.7.10.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Execution log at: /tmp/root/root_20240529203505_1f0cd7e2-d35f-4777-9834-393e39c250c5.log
2024-05-29 20:35:17     Starting to launch local task to process map join;      maximum memory = 477626368
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
2024-05-29 20:35:19     Dump the side-table for tag: 1 with group count: 10 into file: file:/tmp/root/b8f872cc-0994-46c8-a95b-abfe851a5b40/hive_2024-05-29_20-35-05_431_7357886455861394750-1/-local-10002/HashTable-Stage-4/MapJoin-mapfile11--.hashtable
2024-05-29 20:35:19     Uploaded 1 File to: file:/tmp/root/b8f872cc-0994-46c8-a95b-abfe851a5b40/hive_2024-05-29_20-35-05_431_7357886455861394750-1/-local-10002/HashTable-Stage-4/MapJoin-mapfile11--.hashtable (529 bytes)
2024-05-29 20:35:19     End of local task; Time Taken: 2.291 sec.
Execution completed successfully
MapredLocal task succeeded
Launching Job 1 out of 1
Number of reduce tasks is set to 0 since there's no reduce operator
Job running in-process (local Hadoop)
2024-05-29 20:35:22,455 Stage-4 map = 0%,  reduce = 0%
2024-05-29 20:35:23,464 Stage-4 map = 100%,  reduce = 0%
Ended Job = job_local1991539537_0008
Loading data to table universal.transaccion_persona_enriquecida_2
MapReduce Jobs Launched:
Stage-Stage-4:  HDFS Read: 10951320 HDFS Write: 5528372 SUCCESS
Total MapReduce CPU Time Spent: 0 msec
OK
Time taken: 18.159 seconds
hive>

In [ ]:
-- Desde HIVE, verificamos que se hayan insertado los datos en la tabla
SELECT * FROM UNIVERSAL.TRANSACCION_PERSONA_ENRIQUECIDA_2 LIMIT 10;

In [ ]:
hive> SELECT * FROM UNIVERSAL.TRANSACCION_PERSONA_ENRIQUECIDA_2 LIMIT 10;
OK
+-----------------------------------------------+---------------------------------------------------+-------------------------------------------------+----------------------------------------------------+----------------------------------------------------+-----------------------------------------------------+-----------------------------------------------------+----------------------------------------------------------+
| transaccion_persona_enriquecida_2.id_persona  | transaccion_persona_enriquecida_2.nombre_persona  | transaccion_persona_enriquecida_2.edad_persona  | transaccion_persona_enriquecida_2.salario_persona  | transaccion_persona_enriquecida_2.trabajo_persona  | transaccion_persona_enriquecida_2.monto_transaccion | transaccion_persona_enriquecida_2.fecha_transaccion | transaccion_persona_enriquecida_2.id_empresa_transaccion |
+-----------------------------------------------+---------------------------------------------------+-------------------------------------------------+----------------------------------------------------+----------------------------------------------------+-----------------------------------------------------+-----------------------------------------------------+----------------------------------------------------------+
| 18                                            | Owen                                              | 34                                              | 4759.0                                             | Samsung                                            | 1383.0                                              | 2018-01-21                                          | 3                                                        |
| 30                                            | Clayton                                           | 52                                              | 9505.0                                             | HP                                                 | 2331.0                                              | 2018-01-21                                          | 6                                                        |
| 47                                            | Vernon                                            | 35                                              | 7109.0                                             | Sony                                               | 2280.0                                              | 2018-01-21                                          | 2                                                        |
| 28                                            | Stephen                                           | 53                                              | 9469.0                                             | HP                                                 | 730.0                                               | 2018-01-21                                          | 1                                                        |
| 91                                            | Erica                                             | 32                                              | 8934.0                                             | Google                                             | 3081.0                                              | 2018-01-21                                          | 4                                                        |
| 74                                            | Kaitlin                                           | 56                                              | 6515.0                                             | Sony                                               | 2409.0                                              | 2018-01-21                                          | 8                                                        |
| 83                                            | Giselle                                           | 45                                              | 2503.0                                             | Microsoft                                          | 2079.0                                              | 2018-01-21                                          | 5                                                        |
| 48                                            | Illiana                                           | 18                                              | 1454.0                                             | HP                                                 | 2543.0                                              | 2018-01-21                                          | 4                                                        |
| 15                                            | Wanda                                             | 27                                              | 1539.0                                             | Amazon                                             | 1434.0                                              | 2018-01-21                                          | 6                                                        |
| 89                                            | Kelly                                             | 55                                              | 10110.0                                            | Google                                             | 780.0                                               | 2018-01-21                                          | 4                                                        |
+-----------------------------------------------+---------------------------------------------------+-------------------------------------------------+----------------------------------------------------+----------------------------------------------------+-----------------------------------------------------+-----------------------------------------------------+----------------------------------------------------------+
Time taken: 0.641 seconds, Fetched: 10 row(s)
hive>

In [ ]:
-- Verificamos los tipos de datos de sus campos
DESCRIBE UNIVERSAL.TRANSACCION_PERSONA_ENRIQUECIDA_2;

In [ ]:
hive> DESCRIBE UNIVERSAL.TRANSACCION_PERSONA_ENRIQUECIDA_2;
OK
+-------------------------+------------+----------+
|        col_name         | data_type  | comment  |
+-------------------------+------------+----------+
| id_persona              | string     |          |
| nombre_persona          | string     |          |
| edad_persona            | int        |          |
| salario_persona         | double     |          |
| trabajo_persona         | string     |          | <---- Columna añadida 
| monto_transaccion       | double     |          |
| fecha_transaccion       | string     |          |
| id_empresa_transaccion  | string     |          |
+-------------------------+------------+----------+
Time taken: 0.029 seconds, Fetched: 8 row(s)
hive>

##### **Creación e inserción de datos para la tabla temporal `TRANSACCION_PERSONA_EMPRESA_ENRIQUECIDA`**

In [ ]:
DROP TABLE IF EXISTS UNIVERSAL.TRANSACCION_PERSONA_EMPRESA_ENRIQUECIDA;
CREATE TEMPORARY TABLE UNIVERSAL.TRANSACCION_PERSONA_EMPRESA_ENRIQUECIDA(
ID_PERSONA STRING,
NOMBRE_PERSONA STRING,
EDAD_PERSONA INT,
SALARIO_PERSONA DOUBLE,
TRABAJO_PERSONA STRING,
MONTO_TRANSACCION DOUBLE,
FECHA_TRANSACCION STRING,
EMPRESA_TRANSACCION STRING
)
STORED AS PARQUET;

In [ ]:
# Desde consola Hive
hive> DROP TABLE IF EXISTS UNIVERSAL.TRANSACCION_PERSONA_EMPRESA_ENRIQUECIDA;
OK
Time taken: 0.048 seconds
hive> CREATE TEMPORARY TABLE UNIVERSAL.TRANSACCION_PERSONA_EMPRESA_ENRIQUECIDA(
    > ID_PERSONA STRING,
    > NOMBRE_PERSONA STRING,
    > EDAD_PERSONA INT,
    > SALARIO_PERSONA DOUBLE,
    > TRABAJO_PERSONA STRING,
    > MONTO_TRANSACCION DOUBLE,
    > FECHA_TRANSACCION STRING,
    > EMPRESA_TRANSACCION STRING
    > )
    > STORED AS PARQUET;
OK
Time taken: 0.095 seconds
hive>

In [ ]:
-- Copiamos datos del join entre las tablas "TRANSACCION_PERSONA_ENRIQUECIDA_2" y "UNIVERSAL.EMPRESA" a la tabla "UNIVERSAL.TRANSACCION_PERSONA_EMPRESA_ENRIQUECIDA"
INSERT INTO UNIVERSAL.TRANSACCION_PERSONA_EMPRESA_ENRIQUECIDA
SELECT
  T.ID_PERSONA AS ID_PERSONA,
  T.NOMBRE_PERSONA AS NOMBRE_PERSONA,
  T.EDAD_PERSONA AS EDAD_PERSONA,
  T.SALARIO_PERSONA AS SALARIO_PERSONA,
  T.TRABAJO_PERSONA AS TRABAJO_PERSONA,
  T.MONTO_TRANSACCION AS MONTO_TRANSACCION,
  T.FECHA_TRANSACCION AS FECHA_TRANSACCION,
  E.NOMBRE AS EMPRESA_TRANSACCION
FROM
  UNIVERSAL.TRANSACCION_PERSONA_ENRIQUECIDA_2 T
JOIN 
  UNIVERSAL.EMPRESA E
ON 
  T.ID_EMPRESA_TRANSACCION = E.ID;

In [ ]:
# Desde consola Hive
hive> INSERT INTO UNIVERSAL.TRANSACCION_PERSONA_EMPRESA_ENRIQUECIDA
    > SELECT
    >   T.ID_PERSONA AS ID_PERSONA,
    >   T.NOMBRE_PERSONA AS NOMBRE_PERSONA,
    >   T.EDAD_PERSONA AS EDAD_PERSONA,
    >   T.SALARIO_PERSONA AS SALARIO_PERSONA,
    >   T.TRABAJO_PERSONA AS TRABAJO_PERSONA,
    >   T.MONTO_TRANSACCION AS MONTO_TRANSACCION,
    >   T.FECHA_TRANSACCION AS FECHA_TRANSACCION,
    >   E.NOMBRE AS EMPRESA_TRANSACCION
    > FROM
    >   UNIVERSAL.TRANSACCION_PERSONA_ENRIQUECIDA_2 T
    > JOIN
    >   UNIVERSAL.EMPRESA E
    > ON
    >   T.ID_EMPRESA_TRANSACCION = E.ID;
WARNING: Hive-on-MR is deprecated in Hive 2 and may not be available in the future versions. Consider using a different execution engine (i.e. spark, tez) or using Hive 1.X releases.
Query ID = root_20240529204003_81c114a8-aab2-42b7-b941-a0d544df9af3
Total jobs = 1
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/opt/hive/lib/log4j-slf4j-impl-2.6.2.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/opt/hadoop-2.7.4/share/hadoop/common/lib/slf4j-log4j12-1.7.10.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Execution log at: /tmp/root/root_20240529204003_81c114a8-aab2-42b7-b941-a0d544df9af3.log
2024-05-29 20:40:35     Starting to launch local task to process map join;      maximum memory = 477626368
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
2024-05-29 20:40:42     Dump the side-table for tag: 1 with group count: 10 into file: file:/tmp/root/b8f872cc-0994-46c8-a95b-abfe851a5b40/hive_2024-05-29_20-40-03_753_1220691037448172620-1/-local-10002/HashTable-Stage-4/MapJoin-mapfile21--.hashtable
2024-05-29 20:40:42     Uploaded 1 File to: file:/tmp/root/b8f872cc-0994-46c8-a95b-abfe851a5b40/hive_2024-05-29_20-40-03_753_1220691037448172620-1/-local-10002/HashTable-Stage-4/MapJoin-mapfile21--.hashtable (529 bytes)
2024-05-29 20:40:42     End of local task; Time Taken: 7.062 sec.
Execution completed successfully
MapredLocal task succeeded
Launching Job 1 out of 1
Number of reduce tasks is set to 0 since there's no reduce operator
Job running in-process (local Hadoop)
2024-05-29 20:40:45,111 Stage-4 map = 0%,  reduce = 0%
2024-05-29 20:40:49,125 Stage-4 map = 100%,  reduce = 0%
Ended Job = job_local377195465_0009
Loading data to table universal.transaccion_persona_empresa_enriquecida
MapReduce Jobs Launched:
Stage-Stage-4:  HDFS Read: 13858378 HDFS Write: 6981230 SUCCESS
Total MapReduce CPU Time Spent: 0 msec
OK
Time taken: 45.549 seconds
hive>

In [ ]:
-- Desde HIVE, verificamos que se hayan insertado los datos en la tabla
SELECT * FROM UNIVERSAL.TRANSACCION_PERSONA_EMPRESA_ENRIQUECIDA LIMIT 10;

In [ ]:
hive> SELECT * FROM UNIVERSAL.TRANSACCION_PERSONA_EMPRESA_ENRIQUECIDA LIMIT 10;
OK
+----------------------------------------------------+--------------------------------------------------------+------------------------------------------------------+---------------------------------------------------------+---------------------------------------------------------+-----------------------------------------------------------+-----------------------------------------------------------+-------------------------------------------------------------+
| transaccion_persona_empresa_enriquecida.id_persona | transaccion_persona_empresa_enriquecida.nombre_persona | transaccion_persona_empresa_enriquecida.edad_persona | transaccion_persona_empresa_enriquecida.salario_persona | transaccion_persona_empresa_enriquecida.trabajo_persona | transaccion_persona_empresa_enriquecida.monto_transaccion | transaccion_persona_empresa_enriquecida.fecha_transaccion | transaccion_persona_empresa_enriquecida.empresa_transaccion |
+----------------------------------------------------+--------------------------------------------------------+------------------------------------------------------+---------------------------------------------------------+---------------------------------------------------------+-----------------------------------------------------------+-----------------------------------------------------------+-------------------------------------------------------------+
| 18                                                 | Owen                                                   | 34                                                   | 4759.0                                                  | Samsung                                                 | 1383.0                                                    | 2018-01-21                                                | Apple                                                       |
| 30                                                 | Clayton                                                | 52                                                   | 9505.0                                                  | HP                                                      | 2331.0                                                    | 2018-01-21                                                | Google                                                      |
| 47                                                 | Vernon                                                 | 35                                                   | 7109.0                                                  | Sony                                                    | 2280.0                                                    | 2018-01-21                                                | Microsoft                                                   |
| 28                                                 | Stephen                                                | 53                                                   | 9469.0                                                  | HP                                                      | 730.0                                                     | 2018-01-21                                                | Walmart                                                     | 
| 91                                                 | Erica                                                  | 32                                                   | 8934.0                                                  | Google                                                  | 3081.0                                                    | 2018-01-21                                                | Toyota                                                      |
| 74                                                 | Kaitlin                                                | 56                                                   | 6515.0                                                  | Sony                                                    | 2409.0                                                    | 2018-01-21                                                | HP                                                          |
| 83                                                 | Giselle                                                | 45                                                   | 2503.0                                                  | Microsoft                                               | 2079.0                                                    | 2018-01-21                                                | Amazon                                                      |
| 48                                                 | Illiana                                                | 18                                                   | 1454.0                                                  | HP                                                      | 2543.0                                                    | 2018-01-21                                                | Toyota                                                      |
| 15                                                 | Wanda                                                  | 27                                                   | 1539.0                                                  | Amazon                                                  | 1434.0                                                    | 2018-01-21                                                | Google                                                      |
| 89                                                 | Kelly                                                  | 55                                                   | 10110.0                                                 | Google                                                  | 780.0                                                     | 2018-01-21                                                | Toyota                                                      |
+----------------------------------------------------+--------------------------------------------------------+------------------------------------------------------+---------------------------------------------------------+---------------------------------------------------------+-----------------------------------------------------------+-----------------------------------------------------------+-------------------------------------------------------------+
Time taken: 0.336 seconds, Fetched: 10 row(s)
hive>

In [ ]:
-- Verificamos los tipos de datos de sus campos
DESCRIBE UNIVERSAL.TRANSACCION_PERSONA_EMPRESA_ENRIQUECIDA;

In [ ]:
hive> DESCRIBE UNIVERSAL.TRANSACCION_PERSONA_EMPRESA_ENRIQUECIDA;
OK
+----------------------+------------+----------+
|       col_name       | data_type  | comment  |
+----------------------+------------+----------+
| id_persona           | string     |          |
| nombre_persona       | string     |          |
| edad_persona         | int        |          |
| salario_persona      | double     |          |
| trabajo_persona      | string     |          |
| monto_transaccion    | double     |          |
| fecha_transaccion    | string     |          |
| empresa_transaccion  | string     |          | <---- columna añadida 
+----------------------+------------+----------+
Time taken: 0.02 seconds, Fetched: 8 row(s)
hive>

##### **Inserción final de datos a la tabla `TRANSACCION_ENRIQUECIDA`**

In [ ]:
ACTIVAR PARTICIONAMIENTO DINAMICO
---------------------------------

-- Ya lo hicimos durante la creación de la capa LANDING

SET hive.exec.dynamic.partition=true;
SET hive.exec.dynamic.partition.mode=nonstrict;

In [ ]:
CONFIGURAR NUMERO DE PARTICIONES =======================> Por defecto se generan 100 particiones. Pero en el caso de que
--------------------------------                          estemos cargando data historica y se generen mas particiones
SET hive.exec.max.dynamic.partitions=9999;                devolvera un error. Es por eso, que se escribe este comando
SET hive.exec.max.dynamic.partitions.pernode=9999;        dandole un numero xxx de particiones.

In [ ]:
hive> SET hive.exec.max.dynamic.partitions=9999;
hive> SET hive.exec.max.dynamic.partitions.pernode=9999;
hive>

In [ ]:
-- Copiamos datos la tabla "UNIVERSAL.TRANSACCION_PERSONA_EMPRESA_ENRIQUECIDA" a la tabla "UNIVERSAL.TRANSACCION_ENRIQUECIDA"
INSERT OVERWRITE TABLE UNIVERSAL.TRANSACCION_ENRIQUECIDA
PARTITION (FECHA_TRANSACCION)
SELECT
  ID_PERSONA,
  NOMBRE_PERSONA,
  EDAD_PERSONA,
  SALARIO_PERSONA,
  TRABAJO_PERSONA,
  MONTO_TRANSACCION,
  EMPRESA_TRANSACCION,
  FECHA_TRANSACCION
FROM
  UNIVERSAL.TRANSACCION_PERSONA_EMPRESA_ENRIQUECIDA;

In [ ]:
hive> INSERT OVERWRITE TABLE UNIVERSAL.TRANSACCION_ENRIQUECIDA
    > PARTITION (FECHA_TRANSACCION)
    > SELECT
    >   ID_PERSONA,
    >   NOMBRE_PERSONA,
    >   EDAD_PERSONA,
    >   SALARIO_PERSONA,
    >   TRABAJO_PERSONA,
    >   MONTO_TRANSACCION,
    >   EMPRESA_TRANSACCION,
    >   FECHA_TRANSACCION
    > FROM
    >   UNIVERSAL.TRANSACCION_PERSONA_EMPRESA_ENRIQUECIDA;
WARNING: Hive-on-MR is deprecated in Hive 2 and may not be available in the future versions. Consider using a different execution engine (i.e. spark, tez) or using Hive 1.X releases.
Query ID = root_20240529214145_25b193d8-8730-435e-aea9-62671521a2c0
Total jobs = 3
Launching Job 1 out of 3
Number of reduce tasks is set to 0 since there's no reduce operator
Job running in-process (local Hadoop)
2024-05-29 21:41:47,022 Stage-1 map = 0%,  reduce = 0%
2024-05-29 21:41:48,030 Stage-1 map = 100%,  reduce = 0%
Ended Job = job_local508554552_0012
Stage-4 is selected by condition resolver.
Stage-3 is filtered out by condition resolver.
Stage-5 is filtered out by condition resolver.
Moving data to directory hdfs://namenode:8020/user/proyecto/database/universal/transaccion_enriquecida/.hive-staging_hive_2024-05-29_21-41-45_310_3138202531186238916-1/-ext-10000
Loading data to table universal.transaccion_enriquecida partition (fecha_transaccion=null)

Loaded : 3/3 partitions.
         Time taken to load dynamic partitions: 0.409 seconds
         Time taken for adding to write entity : 0.002 seconds
MapReduce Jobs Launched:
Stage-Stage-1:  HDFS Read: 16765954 HDFS Write: 8447656 SUCCESS
Total MapReduce CPU Time Spent: 0 msec
OK
_col0   _col1   _col2   _col3   _col4   _col5   _col6   _col7
Time taken: 3.812 seconds
hive>

In [ ]:
-- Desde HIVE, verificamos que se hayan insertado los datos en la tabla
SELECT * FROM UNIVERSAL.TRANSACCION_ENRIQUECIDA LIMIT 10;

In [ ]:
hive> SELECT * FROM UNIVERSAL.TRANSACCION_ENRIQUECIDA LIMIT 10;
OK
+-------------------------------------+-----------------------------------------+---------------------------------------+------------------------------------------+------------------------------------------+--------------------------------------------+----------------------------------------------+--------------------------------------------+
| transaccion_enriquecida.id_persona  | transaccion_enriquecida.nombre_persona  | transaccion_enriquecida.edad_persona  | transaccion_enriquecida.salario_persona  | transaccion_enriquecida.trabajo_persona  | transaccion_enriquecida.monto_transaccion  | transaccion_enriquecida.empresa_transaccion  | transaccion_enriquecida.fecha_transaccion  |
+-------------------------------------+-----------------------------------------+---------------------------------------+------------------------------------------+------------------------------------------+--------------------------------------------+----------------------------------------------+--------------------------------------------+
| 18                                  | Owen                                    | 34                                    | 4759.0                                   | Samsung                                  | 1383.0                                     | Apple                                        | 2018-01-21                                 |
| 30                                  | Clayton                                 | 52                                    | 9505.0                                   | HP                                       | 2331.0                                     | Google                                       | 2018-01-21                                 |
| 47                                  | Vernon                                  | 35                                    | 7109.0                                   | Sony                                     | 2280.0                                     | Microsoft                                    | 2018-01-21                                 |
| 28                                  | Stephen                                 | 53                                    | 9469.0                                   | HP                                       | 730.0                                      | Walmart                                      | 2018-01-21                                 |
| 91                                  | Erica                                   | 32                                    | 8934.0                                   | Google                                   | 3081.0                                     | Toyota                                       | 2018-01-21                                 |
| 74                                  | Kaitlin                                 | 56                                    | 6515.0                                   | Sony                                     | 2409.0                                     | HP                                           | 2018-01-21                                 |
| 83                                  | Giselle                                 | 45                                    | 2503.0                                   | Microsoft                                | 2079.0                                     | Amazon                                       | 2018-01-21                                 |
| 48                                  | Illiana                                 | 18                                    | 1454.0                                   | HP                                       | 2543.0                                     | Toyota                                       | 2018-01-21                                 |
| 15                                  | Wanda                                   | 27                                    | 1539.0                                   | Amazon                                   | 1434.0                                     | Google                                       | 2018-01-21                                 |
| 89                                  | Kelly                                   | 55                                    | 10110.0                                  | Google                                   | 780.0                                      | Toyota                                       | 2018-01-21                                 |
+-------------------------------------+-----------------------------------------+---------------------------------------+------------------------------------------+------------------------------------------+--------------------------------------------+----------------------------------------------+--------------------------------------------+
Time taken: 0.17 seconds, Fetched: 10 row(s)
hive>

In [ ]:
-- Verificamos los tipos de datos de sus campos
DESCRIBE UNIVERSAL.TRANSACCION_ENRIQUECIDA;

In [ ]:
hive> DESCRIBE UNIVERSAL.TRANSACCION_ENRIQUECIDA;
OK
+--------------------------+------------+----------+
|         col_name         | data_type  | comment  |
+--------------------------+------------+----------+
| id_persona               | int        |          |
| nombre_persona           | string     |          |
| edad_persona             | int        |          |
| salario_persona          | double     |          |
| trabajo_persona          | string     |          |
| monto_transaccion        | double     |          |
| empresa_transaccion      | string     |          |
| fecha_transaccion        | string     |          |
|                          |            |          |
| # Partition Information  |            |          |
| # col_name               | data_type  | comment  |
| fecha_transaccion        | string     |          |
+--------------------------+------------+----------+
Time taken: 0.12 seconds, Fetched: 13 row(s)
hive>

In [ ]:
# Revisamos HDFS
root@0ab30c74254e:/home/hadoop/archivos# hdfs dfs -ls /user/proyecto/database/universal
Found 4 items
drwxr-xr-x   - root supergroup          0 2024-05-29 18:26 /user/proyecto/database/universal/empresa
drwxr-xr-x   - root supergroup          0 2024-05-29 18:22 /user/proyecto/database/universal/persona
drwxr-xr-x   - root supergroup          0 2024-05-29 18:39 /user/proyecto/database/universal/transaccion
drwxr-xr-x   - root supergroup          0 2024-05-29 21:41 /user/proyecto/database/universal/transaccion_enriquecida
root@0ab30c74254e:/home/hadoop/archivos#
root@0ab30c74254e:/home/hadoop/archivos# hdfs dfs -ls /user/proyecto/database/universal/transaccion_enriquecida
Found 3 items
drwxr-xr-x   - root supergroup          0 2024-05-29 21:41 /user/proyecto/database/universal/transaccion_enriquecida/fecha_transaccion=2018-01-21
drwxr-xr-x   - root supergroup          0 2024-05-29 21:41 /user/proyecto/database/universal/transaccion_enriquecida/fecha_transaccion=2018-01-22
drwxr-xr-x   - root supergroup          0 2024-05-29 21:41 /user/proyecto/database/universal/transaccion_enriquecida/fecha_transaccion=2018-01-23
root@0ab30c74254e:/home/hadoop/archivos#
root@0ab30c74254e:/home/hadoop/archivos# hdfs dfs -ls /user/proyecto/database/universal/transaccion_enriquecida/fecha_transaccion=2018-01-21
Found 1 items
-rwxr-xr-x   3 root supergroup     311157 2024-05-29 21:41 /user/proyecto/database/universal/transaccion_enriquecida/fecha_transaccion=2018-01-21/000000_0
root@0ab30c74254e:/home/hadoop/archivos#

##### **Eliminación de tablas temporales**

In [ ]:
DROP TABLE IF EXISTS UNIVERSAL.TRANSACCION_PERSONA_ENRIQUECIDA_1;
DROP TABLE IF EXISTS UNIVERSAL.TRANSACCION_PERSONA_ENRIQUECIDA_2;
DROP TABLE IF EXISTS UNIVERSAL.TRANSACCION_PERSONA_EMPRESA_ENRIQUECIDA;

In [ ]:
hive> DROP TABLE IF EXISTS UNIVERSAL.TRANSACCION_PERSONA_ENRIQUECIDA_1;
OK
Time taken: 0.053 seconds
hive> DROP TABLE IF EXISTS UNIVERSAL.TRANSACCION_PERSONA_ENRIQUECIDA_2;
OK
Time taken: 0.019 seconds
hive> DROP TABLE IF EXISTS UNIVERSAL.TRANSACCION_PERSONA_EMPRESA_ENRIQUECIDA;
OK
Time taken: 0.024 seconds
hive>

#### **`Paso 7` - Crear la capa SMART**

En este paso, los datos que han sido transformados y almacenados en la capa UNIVERSAL son utilizados para realizar análisis avanzados y generar insights valiosos para la organización. Esto puede incluir la aplicación de algoritmos de machine learning, análisis predictivo, análisis de redes sociales, y otras técnicas de procesamiento analítico avanzado para descubrir patrones, tendencias y relaciones ocultas en los datos.

Los resultados de estos análisis son almacenados en la capa SMART del Data Lake. Esta capa contiene datos procesados y enriquecidos que son específicamente diseñados para respaldar la toma de decisiones estratégicas y la generación de valor para la organización. Los insights generados en esta capa pueden ser utilizados por los líderes empresariales y los equipos de análisis para impulsar la innovación, optimizar procesos y mejorar el rendimiento empresarial.

En resumen, los pasos 3 y 4 en la construcción de un Data Lake involucran la transformación de datos en la capa UNIVERSAL para su posterior análisis y generación de insights en la capa SMART. Estos pasos son fundamentales para aprovechar el valor de los datos y convertirlos en acciones que impulsen el éxito empresarial.

- En la capa `SMART` es donde se programa. Red neuronal, reporteria, visualizacion, algo en tiempo real, todo eso va aqui.
- **¿Si no se tienen las 3 capas anteriores a `SMART` podemos realizar BIG DATA?** Claro que si, PERO SIN TENER UN GOBIERNO DE DATOS SOBRE BIG DATA, porque cada desarrollador haria lo que quisiera, quizas algunos carguen los archivos a TEXTO PLANO y desde ahi comiencen a programar, por ejemplo Por tanto, si se tiene un Gobierno de datos basando en un DATA LAKE, se tendra resuelta esa etapa problematica previa al desarrollo de soluciones.

##### **Diagrama de proceso para la obtención de las tablas `TRANSACCION_POR_EDAD`, `TRANSACCION_POR_TRABAJO` y `TRANSACCION_POR_EMPRESA`**

In [ ]:
TRANSACCION                                 TRANSACCION_ENRIQUECIDA                                                            TRANSACCION_POR_EDAD
 ________                                              ________    ID_PERSONA                                 ___________            ________     EDAD_PERSONA     
|__|__|__| ID_PERSONA                                 |__|__|__|   NOMBRE_PERSONA                            |           |          |__|__|__|    CANTIDAD_TRANSACCIONES
|__|__|__| ID_EMPRESA ----.                .--------> |__|__|__|   EDAD_PERSONA                    .-------> | PROCESO 2 | -------> |__|__|__|    SUMA_MONTO_TRANSACCIONES
|__|__|__| MONTO          |                |          |__|__|__|   SALARIO_PERSONA                 |         |___________|          |__|__|__|
           FECHA          |                |                       TRABAJO_PERSONA  ---------------|
PERSONA                   |                |                       MONTO_TRANSACCION               | 
 ________  ID             |           _____|_____                  FECHA_TRANSACCION               |                           TRANSACCION_POR_TRABAJO
|__|__|__| NOMBRE         |          |           |                 EMPRESA_TRANSACCION             |          ___________            ________     TRABAJO_PERSONA
|__|__|__| TELEFONO ------|--------> | PROCESO 1 |                                                 |         |           |          |__|__|__|    CANTIDAD_TRANSACCIONES
|__|__|__| CORREO         |          |___________|                                                 |-------> | PROCESO 3 | -------> |__|__|__|    SUMA_MONTO_TRANSACCIONES
           FECHA_INGRESO  |                                                                        |         |___________|          |__|__|__|
           EDAD           |                                                                        | 
           SALARIO        |                                                                        | 
           ID_EMPRESA     |                                                                        |                                                                           
EMPRESA                   |                                                                        |                           TRANSACCION_POR_EMPRESA      
 ________                 |                                                                        |          ___________            ________    EMPRESA_TRANSACCION
|__|__|__| ID             |                                                                        |         |           |          |__|__|__|   CANTIDAD_TRANSACCIONES
|__|__|__| NOMBRE --------'                                                                        '-------> | PROCESO 4 | -------> |__|__|__|   SUMA_MONTO_TRANSACCIONES
|__|__|__|                                                                                                   |___________|          |__|__|__|


##### `TRANSACCION_POR_EDAD`

In [ ]:
-----------
CREAR TABLA
-----------

DROP TABLE IF EXISTS SMART.TRANSACCION_POR_EDAD;
CREATE TABLE SMART.TRANSACCION_POR_EDAD(
EDAD_PERSONA INT,
CANTIDAD_TRANSACCIONES INT,
SUMA_MONTO_TRANSACCIONES DOUBLE
)
STORED AS PARQUET
LOCATION '/user/proyecto/database/smart/transaccion_por_edad'
TBLPROPERTIES ("parquet.compression"="SNAPPY");

In [ ]:
# Desde consola Hive
hive> DROP TABLE IF EXISTS SMART.TRANSACCION_POR_EDAD;
OK
Time taken: 0.019 seconds
hive> CREATE TABLE SMART.TRANSACCION_POR_EDAD(
    > EDAD_PERSONA INT,
    > CANTIDAD_TRANSACCIONES INT,
    > SUMA_MONTO_TRANSACCIONES DOUBLE
    > )
    > STORED AS PARQUET
    > LOCATION '/user/proyecto/database/smart/transaccion_por_edad'
    > TBLPROPERTIES ("parquet.compression"="SNAPPY");
OK
Time taken: 0.067 seconds
hive>

In [ ]:
-------------------------------
INSERCION DE DATOS EN LA TABLA
-------------------------------

-- Copiamos datos desde la tabla "UNIVERSAL.TRANSACCION_ENRIQUECIDA" hacia la tabla "SMART.TRANSACCION_POR_EDAD"
INSERT OVERWRITE TABLE SMART.TRANSACCION_POR_EDAD
SELECT
  EDAD_PERSONA,
  COUNT(MONTO_TRANSACCION),
  SUM(MONTO_TRANSACCION)
FROM
  UNIVERSAL.TRANSACCION_ENRIQUECIDA
GROUP BY
  EDAD_PERSONA;

In [ ]:
hive> INSERT OVERWRITE TABLE SMART.TRANSACCION_POR_EDAD
    > SELECT
    >   EDAD_PERSONA,
    >   COUNT(MONTO_TRANSACCION),
    >   SUM(MONTO_TRANSACCION)
    > FROM
    >   UNIVERSAL.TRANSACCION_ENRIQUECIDA
    > GROUP BY
    >   EDAD_PERSONA;
WARNING: Hive-on-MR is deprecated in Hive 2 and may not be available in the future versions. Consider using a different execution engine (i.e. spark, tez) or using Hive 1.X releases.
Query ID = root_20240529221102_66c877f1-8446-4d20-9bfd-e74a8605b9a9
Total jobs = 1
Launching Job 1 out of 1
Number of reduce tasks not specified. Estimated from input data size: 1
In order to change the average load for a reducer (in bytes):
  set hive.exec.reducers.bytes.per.reducer=<number>
In order to limit the maximum number of reducers:
  set hive.exec.reducers.max=<number>
In order to set a constant number of reducers:
  set mapreduce.job.reduces=<number>
Job running in-process (local Hadoop)
2024-05-29 22:11:05,993 Stage-1 map = 0%,  reduce = 0%
2024-05-29 22:11:07,002 Stage-1 map = 100%,  reduce = 100%
Ended Job = job_local1783436482_0013
Loading data to table smart.transaccion_por_edad
MapReduce Jobs Launched:
Stage-Stage-1:  HDFS Read: 35369986 HDFS Write: 16896547 SUCCESS
Total MapReduce CPU Time Spent: 0 msec
OK
_col0   _col1   _col2
Time taken: 5.107 seconds
hive>

In [ ]:
-- Desde HIVE, verificamos que se hayan insertado los datos en la tabla
SELECT * FROM SMART.TRANSACCION_POR_EDAD LIMIT 10;

In [ ]:
hive> SELECT * FROM SMART.TRANSACCION_POR_EDAD LIMIT 10;
OK
+------------------------------------+----------------------------------------------+------------------------------------------------+
| transaccion_por_edad.edad_persona  | transaccion_por_edad.cantidad_transacciones  | transaccion_por_edad.suma_monto_transacciones  |
+------------------------------------+----------------------------------------------+------------------------------------------------+
| 18                                 | 4707                                         | 1.0478251E7                                    |
| 19                                 | 9513                                         | 2.1448462E7                                    |
| 22                                 | 11677                                        | 2.613166E7                                     |
| 23                                 | 4670                                         | 1.0415767E7                                    |
| 24                                 | 9379                                         | 2.1185021E7                                    |
| 25                                 | 4621                                         | 1.0358377E7                                    |
| 26                                 | 6891                                         | 1.5612084E7                                    |
| 27                                 | 11766                                        | 2.642914E7                                     |
| 28                                 | 2397                                         | 5419577.0                                      |
| 29                                 | 4607                                         | 1.0278431E7                                    |
+------------------------------------+----------------------------------------------+------------------------------------------------+
Time taken: 0.121 seconds, Fetched: 10 row(s)
hive>

In [ ]:
-- Verificamos los tipos de datos de sus campos
DESCRIBE SMART.TRANSACCION_POR_EDAD;

In [ ]:
hive> DESCRIBE SMART.TRANSACCION_POR_EDAD;
OK
+---------------------------+------------+----------+
|         col_name          | data_type  | comment  |
+---------------------------+------------+----------+
| edad_persona              | int        |          |
| cantidad_transacciones    | int        |          |
| suma_monto_transacciones  | double     |          |
+---------------------------+------------+----------+
Time taken: 0.042 seconds, Fetched: 3 row(s)
hive>

##### `TRANSACCION_POR_TRABAJO`

In [ ]:
-----------
CREAR TABLA
-----------

DROP TABLE IF EXISTS SMART.TRANSACCION_POR_TRABAJO;
CREATE TABLE SMART.TRANSACCION_POR_TRABAJO(
TRABAJO_PERSONA STRING,
CANTIDAD_TRANSACCIONES INT,
SUMA_MONTO_TRANSACCIONES DOUBLE
)
STORED AS PARQUET
LOCATION '/user/proyecto/database/smart/transaccion_por_trabajo'
TBLPROPERTIES ("parquet.compression"="SNAPPY");

In [ ]:
# Desde la consola Hive
hive> DROP TABLE IF EXISTS SMART.TRANSACCION_POR_TRABAJO;
OK
Time taken: 0.017 seconds
hive> CREATE TABLE SMART.TRANSACCION_POR_TRABAJO(
    > TRABAJO_PERSONA STRING,
    > CANTIDAD_TRANSACCIONES INT,
    > SUMA_MONTO_TRANSACCIONES DOUBLE
    > )
    > STORED AS PARQUET
    > LOCATION '/user/proyecto/database/smart/transaccion_por_trabajo'
    > TBLPROPERTIES ("parquet.compression"="SNAPPY");
OK
Time taken: 0.069 seconds
hive>

In [ ]:
-------------------------------
INSERCION DE DATOS EN LA TABLA
-------------------------------

-- Copiamos datos desde la tabla "UNIVERSAL.TRANSACCION_ENRIQUECIDA" hacia la tabla "SMART.TRANSACCION_POR_TRABAJO"
INSERT OVERWRITE TABLE SMART.TRANSACCION_POR_TRABAJO
SELECT
  TRABAJO_PERSONA,
  COUNT(MONTO_TRANSACCION),
  SUM(MONTO_TRANSACCION)
FROM
  UNIVERSAL.TRANSACCION_ENRIQUECIDA
GROUP BY
  TRABAJO_PERSONA;

In [ ]:
hive> INSERT OVERWRITE TABLE SMART.TRANSACCION_POR_TRABAJO
    > SELECT
    >   TRABAJO_PERSONA,
    >   COUNT(MONTO_TRANSACCION),
    >   SUM(MONTO_TRANSACCION)
    > FROM
    >   UNIVERSAL.TRANSACCION_ENRIQUECIDA
    > GROUP BY
    >   TRABAJO_PERSONA;
WARNING: Hive-on-MR is deprecated in Hive 2 and may not be available in the future versions. Consider using a different execution engine (i.e. spark, tez) or using Hive 1.X releases.
Query ID = root_20240529221453_a482322b-b372-452f-858e-ea2a6df48bf6
Total jobs = 1
Launching Job 1 out of 1
Number of reduce tasks not specified. Estimated from input data size: 1
In order to change the average load for a reducer (in bytes):
  set hive.exec.reducers.bytes.per.reducer=<number>
In order to limit the maximum number of reducers:
  set hive.exec.reducers.max=<number>
In order to set a constant number of reducers:
  set mapreduce.job.reduces=<number>
Job running in-process (local Hadoop)
2024-05-29 22:14:55,632 Stage-1 map = 100%,  reduce = 100%
Ended Job = job_local1848114134_0014
Loading data to table smart.transaccion_por_trabajo
MapReduce Jobs Launched:
Stage-Stage-1:  HDFS Read: 36469012 HDFS Write: 16898619 SUCCESS
Total MapReduce CPU Time Spent: 0 msec
OK
_col0   _col1   _col2
Time taken: 2.304 seconds
hive>

In [ ]:
-- Desde HIVE, verificamos que se hayan insertado los datos en la tabla
SELECT * FROM SMART.TRANSACCION_POR_TRABAJO LIMIT 10;

In [ ]:
hive> SELECT * FROM SMART.TRANSACCION_POR_TRABAJO LIMIT 10;
OK
+------------------------------------------+-------------------------------------------------+---------------------------------------------------+
| transaccion_por_trabajo.trabajo_persona  | transaccion_por_trabajo.cantidad_transacciones  | transaccion_por_trabajo.suma_monto_transacciones  |
+------------------------------------------+-------------------------------------------------+---------------------------------------------------+
| Amazon                                   | 32911                                           | 7.3819971E7                                       |
| Apple                                    | 25923                                           | 5.801854E7                                        |
| Google                                   | 30583                                           | 6.8591761E7                                       |
| HP                                       | 21247                                           | 4.7466547E7                                       |
| IBM                                      | 14124                                           | 3.1569729E7                                       |
| Microsoft                                | 33098                                           | 7.4393407E7                                       |
| Samsung                                  | 21160                                           | 4.7443803E7                                       |
| Sony                                     | 20917                                           | 4.7099878E7                                       |
| Toyota                                   | 18979                                           | 4.2356472E7                                       |
| Walmart                                  | 16098                                           | 3.6413955E7                                       |
+------------------------------------------+-------------------------------------------------+---------------------------------------------------+
Time taken: 0.124 seconds, Fetched: 10 row(s)
hive>

In [ ]:
-- Verificamos los tipos de datos de sus campos
DESCRIBE SMART.TRANSACCION_POR_TRABAJO;

In [ ]:
hive> DESCRIBE SMART.TRANSACCION_POR_TRABAJO;
OK
+---------------------------+------------+----------+
|         col_name          | data_type  | comment  |
+---------------------------+------------+----------+
| trabajo_persona           | string     |          |
| cantidad_transacciones    | int        |          |
| suma_monto_transacciones  | double     |          |
+---------------------------+------------+----------+
Time taken: 0.033 seconds, Fetched: 3 row(s)
hive>

##### `TRANSACCION_POR_EMPRESA`

In [ ]:
-----------
CREAR TABLA
-----------

DROP TABLE IF EXISTS SMART.TRANSACCION_POR_EMPRESA;
CREATE TABLE SMART.TRANSACCION_POR_EMPRESA(
EMPRESA_TRANSACCION STRING,
CANTIDAD_TRANSACCIONES INT,
SUMA_MONTO_TRANSACCIONES DOUBLE
)
STORED AS PARQUET
LOCATION '/user/proyecto/database/smart/transaccion_por_empresa'
TBLPROPERTIES ("parquet.compression"="SNAPPY");

In [ ]:
# Desde la consola Hive
hive> DROP TABLE IF EXISTS SMART.TRANSACCION_POR_EMPRESA;
OK
Time taken: 0.016 seconds
hive> CREATE TABLE SMART.TRANSACCION_POR_EMPRESA(
    > EMPRESA_TRANSACCION STRING,
    > CANTIDAD_TRANSACCIONES INT,
    > SUMA_MONTO_TRANSACCIONES DOUBLE
    > )
    > STORED AS PARQUET
    > LOCATION '/user/proyecto/database/smart/transaccion_por_empresa'
    > TBLPROPERTIES ("parquet.compression"="SNAPPY");
OK
Time taken: 0.185 seconds
hive>

In [ ]:
-------------------------------
INSERCION DE DATOS EN LA TABLA
-------------------------------

-- Copiamos datos desde la tabla "UNIVERSAL.TRANSACCION_ENRIQUECIDA" hacia la tabla "SMART.TRANSACCION_POR_EMPRESA"
INSERT OVERWRITE TABLE SMART.TRANSACCION_POR_EMPRESA
SELECT
  EMPRESA_TRANSACCION,
  COUNT(MONTO_TRANSACCION),
  SUM(MONTO_TRANSACCION)
FROM
  UNIVERSAL.TRANSACCION_ENRIQUECIDA
GROUP BY
  EMPRESA_TRANSACCION;

In [ ]:
hive> INSERT OVERWRITE TABLE SMART.TRANSACCION_POR_EMPRESA
    > SELECT
    >   EMPRESA_TRANSACCION,
    >   COUNT(MONTO_TRANSACCION),
    >   SUM(MONTO_TRANSACCION)
    > FROM
    >   UNIVERSAL.TRANSACCION_ENRIQUECIDA
    > GROUP BY
    >   EMPRESA_TRANSACCION;
WARNING: Hive-on-MR is deprecated in Hive 2 and may not be available in the future versions. Consider using a different execution engine (i.e. spark, tez) or using Hive 1.X releases.
Query ID = root_20240529221708_7809400a-e5a8-415d-9216-bcea78ad2e8b
Total jobs = 1
Launching Job 1 out of 1
Number of reduce tasks not specified. Estimated from input data size: 1
In order to change the average load for a reducer (in bytes):
  set hive.exec.reducers.bytes.per.reducer=<number>
In order to limit the maximum number of reducers:
  set hive.exec.reducers.max=<number>
In order to set a constant number of reducers:
  set mapreduce.job.reduces=<number>
Job running in-process (local Hadoop)
2024-05-29 22:17:09,902 Stage-1 map = 100%,  reduce = 0%
2024-05-29 22:17:10,910 Stage-1 map = 100%,  reduce = 100%
Ended Job = job_local2101711162_0015
Loading data to table smart.transaccion_por_empresa
MapReduce Jobs Launched:
Stage-Stage-1:  HDFS Read: 37567260 HDFS Write: 16900297 SUCCESS
Total MapReduce CPU Time Spent: 0 msec
OK
_col0   _col1   _col2
Time taken: 3.089 seconds
hive>

In [ ]:
-- Desde HIVE, verificamos que se hayan insertado los datos en la tabla
SELECT * FROM SMART.TRANSACCION_POR_EMPRESA LIMIT 10;

In [ ]:
hive> SELECT * FROM SMART.TRANSACCION_POR_EMPRESA LIMIT 10;
OK
+----------------------------------------------+-------------------------------------------------+---------------------------------------------------+
| transaccion_por_empresa.empresa_transaccion  | transaccion_por_empresa.cantidad_transacciones  | transaccion_por_empresa.suma_monto_transacciones  |
+----------------------------------------------+-------------------------------------------------+---------------------------------------------------+
| Amazon                                       | 23443                                           | 5.2402645E7                                       |
| Apple                                        | 23575                                           | 5.342402E7                                        |
| Google                                       | 23406                                           | 5.2402979E7                                       |
| HP                                           | 23382                                           | 5.2488235E7                                       |
| IBM                                          | 23581                                           | 5.2589229E7                                       |
| Microsoft                                    | 23472                                           | 5.2680068E7                                       |
| Samsung                                      | 23522                                           | 5.2882909E7                                       |
| Sony                                         | 23616                                           | 5.296073E7                                        |
| Toyota                                       | 23389                                           | 5.2121975E7                                       |
| Walmart                                      | 23654                                           | 5.3221273E7                                       |
+----------------------------------------------+-------------------------------------------------+---------------------------------------------------+
Time taken: 0.091 seconds, Fetched: 10 row(s)
hive>

In [ ]:
-- Verificamos los tipos de datos de sus campos
DESCRIBE SMART.TRANSACCION_POR_EMPRESA;

In [ ]:
hive> DESCRIBE SMART.TRANSACCION_POR_EMPRESA;
OK
+---------------------------+------------+----------+
|         col_name          | data_type  | comment  |
+---------------------------+------------+----------+
| empresa_transaccion       | string     |          |
| cantidad_transacciones    | int        |          |
| suma_monto_transacciones  | double     |          |
+---------------------------+------------+----------+
Time taken: 0.043 seconds, Fetched: 3 row(s)
hive>

#### **`Paso 8` - Ejecución de scripts de solucion por consola**

In [ ]:
                                      --------------------------------------------
                                      EJECUCION DE SCRIPTS DE SOLUCION POR CONSOLA
                                      --------------------------------------------

PROCESO 1:
         ________________________________________________________________________________________________
        |                                                                                                |         
        |   beeline -u jdbc:hive2:// -f /home/main/proceso_1.sql --hiveconf "PARAM_USERNAME=XXXXX"       |
        |________________________________________________________________________________________________|

PROCESO 2:
         ________________________________________________________________________________________________
        |                                                                                                |         
        |   beeline -u jdbc:hive2:// -f /home/main/proceso_2.sql --hiveconf "PARAM_USERNAME=XXXXX"       |
        |________________________________________________________________________________________________|

PROCESO 3:
         ________________________________________________________________________________________________
        |                                                                                                |         
        |   beeline -u jdbc:hive2:// -f /home/main/proceso_3.sql --hiveconf "PARAM_USERNAME=XXXXX"       |
        |________________________________________________________________________________________________|    

PROCESO 4:
         ________________________________________________________________________________________________
        |                                                                                                |         
        |   beeline -u jdbc:hive2:// -f /home/main/proceso_4.sql --hiveconf "PARAM_USERNAME=XXXXX"       |
        |________________________________________________________________________________________________| 

Si en el caso de que existiese MÁS DE 1 PARAMETRO, se agrega a continuacion del primero:
         __________________________________________________________________________________________________________________
        |                                                                                                                  |         
        |   beeline -u jdbc:hive2:// -f /home/main/proceso.sql --hiveconf "PARAM_USERNAME=XXXXX" --hiveconf "PARAM2=XXXX"  |
        |__________________________________________________________________________________________________________________|         
        
          